<a href="https://colab.research.google.com/github/MR-Goldenyt/Argus/blob/main/Argus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# IT7009 Artificial Intelligence - Selected Project Brief #2

## Intelligent Cyberattack Analysis System


## Phase 1: Data Cleaning and Preprocessing


### i. Imports and Data Loading

- **Problem:** every step needs the same libraries, constants and raw data.
- **Decision:** set the seed, the test share, `BIG`, the CPU cores and the figure style once; keep the raw file as `ddf` (never changed) and clean a copy, `df`.
- **Alternatives:** reloading the file in each step would repeat work and risk two different versions of the data.

In [ ]:
# Import Required Libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from imblearn.over_sampling import SMOTENC

# Constants shared across the notebook
DATASET_PATH = "Dataset_Brief_2.csv"  # upload to the Colab session storage first
LABEL_COL = "label"  # target column
SEED, TEST_SIZE = 42, 0.2  # reproducibility and stratified test share
N_CLASSES = 33  # the brief requires all 33 attack classes
BIG = 1e10  # a value below 0 or at least BIG is a placeholder code for "not measured"
# CPU cores the models may use (the cleaning phase trains no model; Phase 2 uses them for the Random Forest, KNN and K-Means).
# Set CORES to 0 to use every core of this computer, or to a positive number to use exactly that many
# (a number above the cores the computer has is capped). Every core makes the computer slow to respond during the
# long Phase 2 runs, so the default leaves some free.
CORES = 2  # cpu cores limit
assert CORES >= 0, "CORES must be 0 (every core) or a positive number of cores"
CORES = (os.cpu_count() or 1) if CORES == 0 else min(CORES, os.cpu_count() or 1)  # the number of cores actually used
FIG_DIR = "figures"  # saved figures go here
SAVE_FIGS = False  # set to True to write every figure to FIG_DIR (the report uses these files)
BG = "white"  # figure background color (light, so the figures print cleanly)
FIG_DPI = 130  # on-screen resolution of every figure (matplotlib default 100)
STYLE = ["default", {"figure.facecolor": BG, "axes.facecolor": BG, "figure.dpi": FIG_DPI}]
plt.rcParams["figure.dpi"] = FIG_DPI  # also for figures drawn outside STYLE
WIDE = (10, 2.8)  # wide, short figures: about 1.7 inches tall at report width

FIG_NO = 0  # figures are numbered in notebook order by finish_fig


def finish_fig(fig, name):
    """Save the figure if SAVE_FIGS is on, number its title, then close it so only display() shows it.

    Args:
        fig: Matplotlib figure to save, number and close.
        name: File name without extension, saved as FIG_DIR/name.png.

    Returns:
        The same figure, ready to pass to display().
    """
    global FIG_NO
    if SAVE_FIGS:  # saved before numbering: the report numbers its own figure captions
        os.makedirs(FIG_DIR, exist_ok=True)
        fig.savefig(f"{FIG_DIR}/{name}.png", dpi=300, facecolor=BG, bbox_inches="tight")  # keep long titles inside the PNG
    FIG_NO += 1
    label = f"Figure {FIG_NO}"
    titled = [ax for ax in fig.axes if ax.get_title()]  # panels with a title (colour bars have none)
    if fig.get_suptitle():  # a figure title: number it
        fig.suptitle(f"{label}: {fig.get_suptitle()}")
    elif len(titled) == 1:  # a single titled panel: number its title
        titled[0].set_title(f"{label}: {titled[0].get_title()}")
    else:  # several titled panels and no figure title: add the number above them
        fig.suptitle(label)
    plt.close(fig)
    return fig

# Raw dataset data frame for future comparisons with clean data
ddf = pd.read_csv(DATASET_PATH, low_memory=False)

# A copy of the raw dataset for cleaning
df = ddf.copy()

# Display the raw dataset
print("Raw dataset shape:", ddf.shape)
pd.set_option("display.max_columns", None) # display all cols
ddf.head()

### ii. Overall Analysis of Raw Data

- **Problem:** the data-quality issues must be found before anything is fixed.
- **Decision:** count the features and column types, then read `info()` and `describe()`: they reveal text-typed numeric columns, missing values and impossible extremes.
- **Alternatives:** looking at a few rows by eye would miss problems that only show in the column summaries.

In [ ]:
# Feature count and column types as read
print("Number of feature columns:", ddf.shape[1] - 1)  # every column except the label
print(f"\nColumns per type as read (including the label):\n{ddf.dtypes.value_counts().to_string()}\n")  # to_string drops the pandas footer

# Brief Information
ddf.info()

# Statistical Summary
ddf.describe(include="all").T

### iii. Categorical, Temporal, and Identifier Audit

- **Problem:** identifiers and timestamps let a model memorise records instead of learning attacks (leakage).
- **Decision:** search by name, by datetime type and by uniqueness; confirm that `label` is the only text-categorical variable. Hidden timing is screened later, in step x.
- **Alternatives:** checking by eye could miss a renamed identifier; dropping high-cardinality columns would remove real measurements.

In [ ]:
# Identifier / temporal check: by name, by dtype and by uniqueness (an ID is unique on every row)
name_hits = [c for c in df.columns if any(k in c.lower() for k in ("id", "date", "time"))]
datetime_cols = df.select_dtypes("datetime").columns.to_list()
unique_cols = df.columns[df.nunique() == len(df)].to_list()
print("Identifier/date-like column names:", name_hits or "none")
print("Datetime-typed columns:", datetime_cols or "none")
print("Columns unique on every row:", unique_cols or "none")

# Categorical check: the target is the only text-categorical variable
print("\nCategorical columns:", [LABEL_COL])
print("Target classes:", df[LABEL_COL].nunique(), "| Missing labels:", df[LABEL_COL].isnull().sum())

### iv. Formatting and Feature Consistency Test

- **Problem:** inconsistent column names (one with a typo) and possible label spelling variants.
- **Decision:** standardise names to `snake_case`; merge labels that differ only in case, spaces, `-` or `_`; assert the 33 classes of the brief.
- **Alternatives:** renaming by hand is error-prone; leaving variants would split one attack into several classes.

In [ ]:
# ======== Feature columns rename map (keys are case-sensitive) ========
RENAME_FIXES = {
    "Magnitue": "magnitude",
    "Tot": "total"
}

# ======== Function Definitions ========
def snake(name):
    """Trim whitespace and convert a column name to lowercase snake_case.

    Args:
        name: Original column name, e.g. "Protocol Type".

    Returns:
        The snake_case name, e.g. "protocol_type".
    """
    return "_".join(name.split()).lower()

def standardize_columns(df: pd.DataFrame, rename_map: dict) -> dict :
    """
    Renames DataFrame columns by applying substring replacements followed by
    a snake_case transformation.
    
    Modifies the DataFrame in-place and returns the old-new mapping.
    
    Args:
        df: The Pandas DataFrame to modify.
        rename_map: A dict of {'old_substring': 'new_substring'} to replace first.
    
    Returns:
        A dict of {original_name: new_name} for columns that changed.
    """
    names_before = list(df.columns)
    processed_names = []

    for col in names_before:
        new_name = col
        # Step 1 Substring replacement
        if rename_map:
            for old_sub, new_sub in rename_map.items():
                new_name = new_name.replace(old_sub, new_sub)
        
        # Step 2 Convert to Snake case 
        new_name = snake(new_name)
        processed_names.append(new_name)

    df.columns = processed_names
    return {a: b for a, b in zip(names_before, processed_names) if a != b}

# ======== Standardise the feature column names ========
names_before = df.columns.to_list()
renamed = standardize_columns(df, RENAME_FIXES)
assert df.columns.is_unique, "Renaming created duplicate column names"
print(f"Renamed {len(renamed)} of {len(names_before)} columns:\n{renamed}")
# Extract new feature columns
feature_cols = [c for c in df.columns if c != LABEL_COL]

# ======== Normalise label text ========
classes_before = df[LABEL_COL].nunique()
labels = df[LABEL_COL].str.strip()
key = labels.str.casefold().str.replace(r"[\s_\-]+", "", regex=True)  # ignores case, spaces, "-" and "_"
canonical = labels.groupby(key).agg(lambda s: s.mode()[0])  # most common spelling names the class
variants = sorted(set(labels) - set(canonical))
df[LABEL_COL] = key.map(canonical)
print(f"\nClasses: {classes_before} -> {df[LABEL_COL].nunique()} (brief states {N_CLASSES}) | variant spellings merged: {variants or 'none'}")
assert df[LABEL_COL].nunique() == N_CLASSES, "Label normalisation merged or split attack classes"

# ======== Print the first 5 rows ========
df.head()

### v. Missing Values and Incorrect Data Types

- **Problem:** text tokens (`invalid`, `unknown`) keep numeric columns as text, and placeholder codes (below 0 or at least `BIG`) pose as real values.
- **Decision:** convert every feature to numbers and turn the tokens and the codes into `NaN`; print each code per column before and after.
- **Alternatives:** keeping the codes would distort means and scales; dropping rows with one bad cell would lose whole flows.

In [ ]:
# ======== Function Definitions ========
def coerce_numeric(raw):
    """Convert every column to numeric and collect the values that could not be parsed.

    Args:
        raw: DataFrame of columns read as text (object dtype).

    Returns:
        A tuple (numeric, bad_values): the converted DataFrame, with unparseable entries as NaN,
        and an array of the original values that failed to parse.
    """
    numeric = raw.apply(pd.to_numeric, errors="coerce")
    bad = raw.notna() & numeric.isna()  # had a value but became NaN
    return numeric, raw.to_numpy()[bad.to_numpy(dtype=bool)]


def print_col_types(cols, before, after):
    """Print each column with its dtype before and after conversion.

    Args:
        cols: Column names to list, numbered from 1.
        before: Series of dtypes before conversion, indexed by column name.
        after: Series of dtypes after conversion, indexed by column name.
    """
    pad = len(str(len(cols)))  # zero-pad the index
    width = max(map(len, cols), default=0)  # align the dtype column
    for i, col in enumerate(cols, 1):
        print(f"  {i:0{pad}d}  {col:<{width}}  {before[col]} -> {after[col]}")


def before_after_bars(panels, title, filename):
    """Draw one before/after bar pair per panel, with the panels side by side.

    Args:
        panels: List of (panel title, y-axis label, value before, value after).
        title: Figure title.
        filename: File name passed to finish_fig.

    Returns:
        The finished figure.
    """
    with plt.style.context(STYLE):
        fig, axes = plt.subplots(1, len(panels), figsize=WIDE)
        for ax, (name, ylabel, before, after) in zip(np.atleast_1d(axes), panels):
            ax.bar(["Before", "After"], [before, after], color=["indianred", "seagreen"], width=0.6)
            ax.bar_label(ax.containers[0], fmt="{:,.0f}", padding=2, fontweight="bold")
            ax.set(title=name, ylabel=ylabel, ylim=(0, max(before, after) * 1.25 or 1))  # headroom for the labels
            ax.grid(axis="y", linestyle="--", alpha=0.2)
            ax.set_axisbelow(True)
        fig.suptitle(title)
        fig.tight_layout()
    return finish_fig(fig, filename)


# ======== Before ========
# 1. Feature columns stored as text, and the text tokens that keep them from being numbers
object_cols_before = df[feature_cols].select_dtypes(exclude="number").columns.to_list()
raw = df[object_cols_before]
dtypes_before = raw.dtypes
missing_values_before = df.isna().to_numpy().sum()
numeric, non_numbers_before = coerce_numeric(raw)
unique_non_numbers_before = pd.unique(non_numbers_before)

# 2. Placeholder codes: a number below 0 or of at least BIG stands for "not measured" (no real feature is negative)
as_numbers = df[feature_cols].copy()
as_numbers[object_cols_before] = numeric  # the values as numbers, so the codes can be compared
codes = (as_numbers < 0) | (as_numbers >= BIG)
code_cells = as_numbers[codes].stack()  # one entry per placeholder cell: (row, column) -> code
placeholders = code_cells.value_counts()  # cells per code
code_table = code_cells.groupby([code_cells.index.get_level_values(1), code_cells]).size()  # cells per column and code
code_table = code_table.rename_axis(["column", "code"]).rename("cells").reset_index()

print("======== Before ========")
print(f"Feature columns stored as text: {len(object_cols_before)} of {len(feature_cols)}")
print(f"Text tokens in them: {len(non_numbers_before)} cells, values {list(unique_non_numbers_before)}")
print(f"Placeholder codes (a number < 0 or >= {BIG:.0e} that means 'not measured'): {int(placeholders.sum())} cells "
      f"in {code_table['column'].nunique()} columns")
print(f"Missing cells: {missing_values_before}")
print("Table: every placeholder code found, with its column and the number of cells that hold it")
display(code_table.assign(code=code_table["code"].map("{:,.0f}".format)))  # codes as whole numbers

# ======== Cleaning ========
df[object_cols_before] = numeric  # text tokens become NaN
df[feature_cols] = df[feature_cols].mask(codes)  # placeholder codes become NaN

# ======== After ========
missing_values_after = df.isna().to_numpy().sum()
object_cols_after = df[feature_cols].select_dtypes(exclude="number").columns.to_list()
codes_left = int(((df[feature_cols] < 0) | (df[feature_cols] >= BIG)).to_numpy().sum())
print("\n======== After ========")
print(f"Feature columns stored as text: {len(object_cols_after)} | placeholder codes left: {codes_left}")
print(f"Missing cells: {missing_values_before} -> {missing_values_after} "
      f"(+{missing_values_after - missing_values_before}: the text tokens and placeholder codes, now empty)")
print("Columns converted (dtype before -> after):")
print_col_types(object_cols_before, dtypes_before, df[object_cols_before].dtypes)

# ======== Conclusion ========
print("\n======== Conclusion ========")
print(f"All {len(feature_cols)} feature columns are numeric, and every 'not measured' value is an empty cell, "
      "handled in step vii (empty rows) and step ix (imputation).")

# ======== Figures ========
display(before_after_bars(
    [("Feature columns stored as text", "Columns", len(object_cols_before), len(object_cols_after)),
     ("Missing cells (filled later, steps vii and ix)", "Cells", missing_values_before, missing_values_after)],
    "Text columns converted to numbers; tokens and placeholder codes become empty cells", "fig1_types_missing"))


### vi. Removing Duplicate Rows

- **Problem:** exact duplicates over-weight some flows and could appear in both training and test.
- **Decision:** remove them after the type conversion and before the split, keeping the first copy.
- **Alternatives:** removing every copy would delete real flows; weighting them keeps the over-weighting.

In [ ]:
# ======== Function Definitions ========
def duplicates_figure(rows_before, rows_after, removed, class_sizes, filename):
    """Draw the rows before and after removing duplicates next to the duplicates removed from each class.

    Args:
        rows_before: Rows before removing duplicates.
        rows_after: Rows after removing duplicates.
        removed: Series of removed row counts, indexed by class (only the classes that lost rows).
        class_sizes: Series of class sizes before removal, for the percentage labels.
        filename: File name passed to finish_fig.

    Returns:
        The finished figure.
    """
    pct = removed / class_sizes[removed.index] * 100  # share of each class that was removed
    with plt.style.context(STYLE):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 0.25 * len(removed) + 1.8), gridspec_kw={"width_ratios": [1, 2]})
        # (a) Rows before vs after
        ax1.bar(["Before", "After"], [rows_before, rows_after], color=["indianred", "seagreen"], width=0.6)
        ax1.bar_label(ax1.containers[0], fmt="{:,.0f}", padding=2, fontweight="bold")
        ax1.set(title="(a) Rows", ylabel="Rows", ylim=(0, rows_before * 1.2))
        # (b) Duplicates removed per class: a stem and a dot per class, labelled with the share of the class
        ax2.hlines(removed.index, 0, removed, color="indianred", linewidth=2)
        ax2.plot(removed, removed.index, "o", color="indianred", markersize=6)
        for cls, n, p in zip(removed.index, removed, pct):
            ax2.annotate(f"{n} ({p:.2f}%)", (n, cls), xytext=(6, 0), textcoords="offset points", va="center", fontsize=7)
        ax2.set(title="(b) Duplicates removed per class", xlabel="Rows removed", xlim=(0, removed.max() * 1.4))
        ax2.tick_params(axis="y", labelsize=7)
        for ax in (ax1, ax2):
            ax.grid(linestyle="--", alpha=0.2)
            ax.set_axisbelow(True)
        fig.suptitle("Exact duplicate rows removed before the split")
        fig.tight_layout()
    return finish_fig(fig, filename)


# ======== Before ========
rows_before = len(df)
class_sizes = df[LABEL_COL].value_counts()  # rows per class before removing duplicates
duplicates_found = int(df.duplicated().sum())  # rows identical to an earlier row in every column
print("======== Before ========")
print(f"Rows: {rows_before} | exact duplicate rows (identical in every column, label included): {duplicates_found}")

# ======== Cleaning ========
df = df.drop_duplicates().reset_index(drop=True)  # keep the first copy of each row

# ======== After ========
rows_removed = rows_before - len(df)
assert not df.duplicated().any(), "Duplicates remain"
# Rows removed from each class (only the classes that lost rows), largest on top of the chart
removed_per_class = (class_sizes - df[LABEL_COL].value_counts()).loc[lambda s: s > 0].sort_values(kind="stable")
print("\n======== After ========")
print(f"Rows: {rows_before} -> {len(df)} | duplicates left: {int(df.duplicated().sum())}")
print(f"Classes that lost a duplicate: {len(removed_per_class)} of {len(class_sizes)} "
      f"(largest loss: {removed_per_class.max()} rows of {removed_per_class.idxmax()})")

# ======== Conclusion ========
print("\n======== Conclusion ========")
print(f"{rows_removed} duplicate rows ({rows_removed / rows_before:.2%} of the data) were removed before the split, "
      "so no row can appear in both the training and the test set.")

# ======== Figures ========
if rows_removed:
    display(duplicates_figure(rows_before, len(df), removed_per_class, class_sizes, "fig3_duplicates"))


### vii. Missing Values: Detection and Removal of Empty Rows

- **Problem:** a few rows are almost entirely empty; most incomplete rows miss only one or two values.
- **Decision:** remove only the rows with more than half of their features missing; impute the rest after the split (step ix).
- **Alternatives:** dropping every incomplete row loses valid flows; filling an empty row invents a flow.

In [ ]:
ROW_MISSING_LIM = 0.5  # remove a row when more than this share of its features is missing

# ======== Before ========
# Count the missing cells of every row
gaps = df[feature_cols].isnull()  # True where a cell is missing
per_row = gaps.sum(axis=1)  # missing cells in each row
empty = gaps.mean(axis=1) > ROW_MISSING_LIM  # rows that are mostly empty
missing_before = int(gaps.to_numpy().sum())  # total missing cells

# Where the gaps are
cols_with_gaps = (gaps.sum() > 0).sum()  # columns with a missing cell
rows_with_gaps = (per_row > 0).sum()  # rows with a missing cell
rows_by_gaps = per_row[per_row > 0].value_counts().sort_index()  # rows per number of missing features
print("======== Before ========")
print(f"Missing cells: {missing_before} in {cols_with_gaps} of {len(feature_cols)} columns | rows with a gap: {rows_with_gaps}")
print("Rows by number of missing features (missing features: rows):", rows_by_gaps.to_dict())
print(f"Rows with more than {ROW_MISSING_LIM:.0%} of their features missing: {empty.sum()}")

# ======== Cleaning ========
# Drop only the mostly-empty rows; small gaps are imputed after the split (step ix)
empty_labels = df.loc[empty, LABEL_COL].value_counts()  # classes of the removed rows
df = df[~empty].reset_index(drop=True)  # keep the other rows

# ======== After ========
# Missing cells left for step ix
missing_after = int(df[feature_cols].isnull().to_numpy().sum())
assert df[LABEL_COL].nunique() == N_CLASSES, "Removing the empty rows emptied a class"
print("\n======== After ========")
print(f"Rows: {len(gaps)} -> {len(df)} | missing cells: {missing_before} -> {missing_after}")
print(f"Removed rows per class: {empty_labels.to_dict()} | classes left: {df[LABEL_COL].nunique()} of {N_CLASSES}")

# ======== Conclusion ========
print("\n======== Conclusion ========")
print(f"Only the {empty.sum()} almost-empty rows were removed and no class was lost; the {missing_after} scattered "
      "missing cells are filled after the split (step ix), from training values only.")

# ======== Figures ========
# Plot: (a) how many features each incomplete row is missing (red = removed), (b) missing cells before vs after
is_removed = rows_by_gaps.index > ROW_MISSING_LIM * len(feature_cols)  # True for the bars of removed rows
with plt.style.context(STYLE):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=WIDE, gridspec_kw={"width_ratios": [3, 2]})

    # (a) Rows by number of missing features
    ax1.bar(rows_by_gaps.index.astype(str), rows_by_gaps, color=np.where(is_removed, "indianred", "#4c9be8"), width=0.6)
    ax1.bar_label(ax1.containers[0], padding=2, fontweight="bold")  # count on top of each bar
    ax1.set(title="(a) Rows by number of missing features", xlabel="Missing features in the row", ylabel="Rows")
    ax1.set_ylim(0, rows_by_gaps.max() * 1.15)  # headroom for the labels

    # (b) Missing cells before vs after
    ax2.bar(["Before", "After"], [missing_before, missing_after], color=["indianred", "seagreen"], width=0.6)
    ax2.bar_label(ax2.containers[0], fmt="{:,.0f}", padding=2, fontweight="bold")
    ax2.set(title="(b) Missing cells", ylabel="Cells")
    ax2.set_ylim(0, missing_before * 1.15)

    # Same light grid on both panels
    for ax in (ax1, ax2):
        ax.grid(axis="y", linestyle="--", alpha=0.2)
        ax.set_axisbelow(True)
    fig.suptitle("Missing values per row (red rows are removed) and missing cells before vs after")
    fig.tight_layout()
display(finish_fig(fig, "fig5_empty_rows"))

### viii. Class Distribution and Stratified Train/Test Split

- **Problem:** strongly imbalanced classes, and a timing feature (`iat`) that appears to record when a flow was captured rather than the traffic.
- **Decision:** stratified 80/20 split with a fixed seed, after the row fixes and before anything is fitted; remove `iat` right after the split (evidence in step x.2). `flow_duration` stays (it varies widely inside the classes and does not follow the capture time), and so does `duration` (its values look like the packet time-to-live).
- **Alternatives:** a random split does not keep the class shares; removing `iat` later would let values be fitted on it first.

In [ ]:
# ======== Function Definitions ========
def barh(data, title, xlabel, filename, logx=False):
    """Draw horizontal bars with the largest row on top; a DataFrame gives grouped bars.

    Args:
        data: Series (one bar per row) or DataFrame (one bar per column in each row).
        title: Figure title.
        xlabel: Label of the x-axis.
        filename: File name passed to finish_fig.
        logx: Use a log scale on the x-axis, for counts that span several orders of magnitude.

    Returns:
        The finished figure.
    """
    colors = ["#4c9be8", "seagreen"]  # first and second bar of each row
    bars_per_row = 1 + (data.ndim == 2)  # a DataFrame draws two bars per row
    with plt.style.context(STYLE):
        ax = data.iloc[::-1].plot.barh(  # reversed so the largest row is on top
            figsize=(9, 0.3 * len(data) * bars_per_row + 1.5),  # height grows with the rows
            logx=logx,
            width=0.8,
            color=colors if data.ndim == 2 else colors[0],
        )
        # Titles, small class names and a light grid
        ax.set(title=title, xlabel=xlabel, ylabel="")
        ax.tick_params(axis="y", labelsize=8)
        ax.grid(axis="x", linestyle="--", alpha=0.2)
        ax.set_axisbelow(True)
        ax.figure.tight_layout()
    return finish_fig(ax.figure, filename)


# ======== Before ========
# Class sizes: a stratified split needs at least 2 rows in every class
class_counts = df[LABEL_COL].value_counts()
assert class_counts.min() >= 2, "Stratified splitting needs at least 2 rows per class"

imbalance = class_counts.max() / class_counts.min()  # largest class size / smallest class size
print("======== Before ========")
print(f"Rows: {len(df)} | classes: {len(class_counts)} | features: {len(feature_cols)}")
print(f"Largest class: {class_counts.idxmax()} ({class_counts.max()} rows) | "
      f"smallest: {class_counts.idxmin()} ({class_counts.min()} rows) | imbalance ratio {imbalance:.1f}:1")

# ======== Cleaning ========
# 80/20 split that keeps every class's share the same on both sides
X_train, X_test, y_train, y_test = train_test_split(
    df[feature_cols],
    df[LABEL_COL],
    test_size=TEST_SIZE,
    stratify=df[LABEL_COL],
    random_state=SEED,
)

# ======== Cleaning 2: the capture timing removed right after the split ========
# iat appears to record when the flow was captured, not the traffic (evidence in step x.2); flow_duration is kept
SESSION_FEATURES = ["iat"]
session_missing = int(X_train[SESSION_FEATURES].isna().to_numpy().sum() + X_test[SESSION_FEATURES].isna().to_numpy().sum())
X_train, X_test = X_train.drop(columns=SESSION_FEATURES), X_test.drop(columns=SESSION_FEATURES)
feature_cols = X_train.columns.to_list()  # every later step works on the remaining features

# ======== After ========
# Largest difference in any class's share between train and test (close to 0 = well stratified)
gap = (y_train.value_counts(normalize=True) - y_test.value_counts(normalize=True)).abs().max()
print("\n======== After ========")
print(f"Training rows: {len(X_train)} | test rows: {len(X_test)} | every class in both: "
      f"{y_train.nunique() == y_test.nunique() == len(class_counts)}")
print(f"Largest difference in a class's share between train and test: {gap:.4%}")
print(f"Capture timing removed right after the split: {', '.join(SESSION_FEATURES)} (with its {session_missing} missing "
      f"cells) | features left: {len(feature_cols)}")

# ======== Conclusion ========
print("\n======== Conclusion ========")
print("The test set keeps the class mix of the data and is set aside before anything is learned from the values.")

# ======== Figures ========
# One bar per class, largest first, on a log axis so the smallest classes stay visible
with plt.style.context(STYLE):
    fig, ax = plt.subplots(figsize=(WIDE[0], WIDE[1] + 1))  # extra height for the rotated class names
    ax.bar(class_counts.index, class_counts, color="#4c9be8", width=0.8)
    ax.set_yscale("log")
    ax.set(title="Cyberattack class distribution (log scale)", ylabel="Rows")
    ax.tick_params(axis="x", labelrotation=90, labelsize=7)  # vertical class names
    ax.grid(axis="y", linestyle="--", alpha=0.2)
    ax.set_axisbelow(True)
    fig.tight_layout()
display(finish_fig(fig, "fig6_class_distribution"))

### ix. Imputing the Remaining Missing Values

- **Problem:** a few scattered missing cells remain in both parts.
- **Decision:** fill them with the training median (continuous) or mode (0/1 indicators); repair any filled value that breaks a flow rule and assert the rules again.
- **Alternatives:** dropping rows loses flows; the mean is pulled by extremes; model-based imputation is overkill for so few cells.

In [ ]:
# ======== Function Definitions ========
def rule_breaks(part, tol=0.0):
    """Check every row against the three flow rules that the raw data always obeys.

    Args:
        part: Rows in original units (a split part, a cross-validation fold or new SMOTE rows).
        tol: Relative tolerance on min <= avg <= max, for values that went through a back-transformation.

    Returns:
        A boolean DataFrame with one column per rule (True = the row breaks that rule).
    """
    tcp_flags = [c for c in part if c.endswith("_flag_number")]  # the TCP flags still in the data
    not_tcp = (part["udp"] == 1) | (part["icmp"] == 1)  # UDP and ICMP flows
    out_of_order = (part["min"] > part["avg"] * (1 + tol) + tol) | (part["avg"] > part["max"] * (1 + tol) + tol)
    return pd.DataFrame({
        "min <= avg <= max": out_of_order,
        "one protocol": part[["tcp", "udp", "icmp"]].sum(axis=1) > 1,  # marked as more than one of tcp/udp/icmp
        "no TCP flag on UDP/ICMP": not_tcp & (part[tcp_flags] == 1).any(axis=1),
    })


def impute(part, medians, modes):
    """Fill the missing cells of one part with training values, then repair imputed values that break a flow rule.

    Args:
        part: DataFrame filled in place (the training part, the test part or a cross-validation fold).
        medians: Fill value of each continuous column, learned from training rows.
        modes: Fill value of each 0/1 column, learned from training rows.

    Returns:
        The number of rows whose imputed values were repaired.
    """
    filled = part.isna()  # the cells about to be imputed
    part[medians.index] = part[medians.index].fillna(medians)
    part[modes.index] = part[modes.index].fillna(modes)
    snapshot = part.copy()  # to count the repaired rows

    # An imputed value must obey the same flow rules as the observed ones: repair it from the flow's own values
    bad_avg = filled["avg"] & ((part["avg"] < part["min"]) | (part["avg"] > part["max"]))
    part.loc[bad_avg, "avg"] = part.loc[bad_avg, "avg"].clip(part.loc[bad_avg, "min"], part.loc[bad_avg, "max"])
    bad_min = filled["min"] & (part["min"] > part["avg"])  # an imputed minimum above the average
    part.loc[bad_min, "min"] = part.loc[bad_min, "avg"]
    bad_max = filled["max"] & (part["max"] < part["avg"])  # an imputed maximum below the average
    part.loc[bad_max, "max"] = part.loc[bad_max, "avg"]
    for p in ("tcp", "udp", "icmp"):  # an imputed protocol flag must not add a second protocol
        part.loc[filled[p] & (part[["tcp", "udp", "icmp"]].sum(axis=1) > 1), p] = 0
    not_tcp = (part["udp"] == 1) | (part["icmp"] == 1)
    for f in [c for c in part if c.endswith("_flag_number")]:  # an imputed TCP flag must be 0 on a UDP/ICMP flow
        part.loc[filled[f] & not_tcp, f] = 0
    return int((part != snapshot).any(axis=1).sum())


# ======== Before ========
# 0/1 indicators and continuous columns, decided on the training set
BINARY = [c for c in feature_cols if set(X_train[c].dropna().unique()) <= {0, 1}]
CONT = [c for c in feature_cols if c not in BINARY]

# Mode imputation relies on every indicator holding only 0 or 1 in all rows, not just in the training rows
indicators = BINARY
bad_states = [c for c in indicators if not df[c].dropna().isin([0, 1]).all()]
assert not bad_states, f"Indicators with values other than 0/1: {bad_states}"

# The raw rows obey the flow rules, so imputed values must too
raw_breaks = rule_breaks(df).sum()  # raw rows breaking each rule
assert raw_breaks.sum() == 0, "Fields that contradict each other"

# Why the median: the feature whose mean is pulled furthest from its median
positive = X_train[CONT].loc[:, X_train[CONT].median() > 0]  # a zero median gives no ratio
example = (positive.mean() / positive.median()).idxmax()
mean_vs_median = {"feature": example, "mean": positive[example].mean(), "median": positive[example].median()}

missing = X_train.isnull().sum()  # missing cells per training column
missing_test = int(X_test.isnull().to_numpy().sum())
imputed_cells = int(missing.sum()) + missing_test  # cells this step fills, both parts
print("======== Before ========")
print(f"Missing cells: {missing.sum()} in the training part, {missing_test} in the test part "
      f"({(missing > 0).sum()} of {len(missing)} columns have a gap)")
print(f"Columns: {len(BINARY)} 0/1 indicators (filled with the mode) | {len(CONT)} continuous (filled with the median)")
print(f"Why the median: {example} has a training mean of {mean_vs_median['mean']:,.0f} but a median of "
      f"{mean_vs_median['median']:,.0f}, because a few extreme flows pull the mean")
print("Flow rules the raw rows obey (rows breaking each):", raw_breaks.to_dict())

# ======== Cleaning ========
# Fill values come from the training set only (median: robust to extremes; mode: keeps flags at 0 or 1)
medians = X_train[CONT].median()
modes = X_train[BINARY].mode().iloc[0]
repaired_rows = sum(impute(part, medians, modes) for part in (X_train, X_test))
for part in (X_train, X_test):
    assert not rule_breaks(part).to_numpy().any(), "An imputed value breaks a flow rule"

# ======== After ========
print("\n======== After ========")
print(f"Missing cells left: {X_train.isnull().sum().sum()} in the training part, {X_test.isnull().sum().sum()} in the test part")
print(f"Imputed rows repaired because a filled value broke a flow rule: {repaired_rows} (every rule holds again)")
stage_ix = (X_train.copy(), y_train.copy())  # the training data after this step, for step xvi

# ======== Conclusion ========
print("\n======== Conclusion ========")
print("Every gap is filled with a value learned from the training part only, so the test part cannot influence training.")

# ======== Figures ========
# The 15 training features with the most missing cells before imputation
display(barh(missing[missing > 0].sort_values(ascending=False).head(15),
             "Missing cells per feature before imputation (train, most affected)",
             "Number of missing cells", "fig7_missing_per_feature"))

### x. Removing Irrelevant and Leaking Features

- **Problem:** near-constant features add nothing, and some features may leak the class.
- **Decision:** review near-constant features (keep one only if its rare values mark one class); screen for leakage with value purity and drop recording stand-ins (x.2: fixed per batch, the same shift in every class, or hidden in a product); remove rows made identical (x.3); list weak and correlated features for Phase 2 (x.4). Training part only, no model.
- **Alternatives:** a variance threshold drops class-specific flags; PCA hides which features matter.

In [ ]:
NEAR_CONST = 0.99  # one value in at least this share of rows = near-constant
MIN_SHARE, MIN_RARE = 0.5, 5  # keep it if at least 5 rare rows exist and half of them share one class
LEAK_PURITY = 0.70  # value purity at or above this = leakage suspect
PURITY_BINS = 100  # bins per feature, each holding the same number of rows
PAIR_TOP = 6  # pairs are formed from this many best single features
BATCH_STEP = 1e7  # a suspect's raw values rounded to this step give its recording batches
SHIFT_SPREAD, SHIFT_MIN = 0.05, 0.05  # recording shift: all class medians move by one factor (spread <= 5%) of at least 5%
HIDDEN_CORR = 0.99  # hidden stand-in: correlation with another feature times a stand-in at or above this
dropped = dict.fromkeys(SESSION_FEATURES, "capture-session timing (removed after the split)")  # feature -> reason it was removed


# ======== Function Definitions ========
def drop(cols, reason):
    """Record why features are removed and return the training set without them.

    Args:
        cols: Feature names to remove.
        reason: Reason stored in `dropped`.

    Returns:
        X_train without the given columns.
    """
    dropped.update(dict.fromkeys(cols, reason))
    return X_train.drop(columns=list(cols))


def rare_profile(col):
    """Describe the rows where a near-constant feature differs from its usual value.

    Args:
        col: Name of a training feature.

    Returns:
        The share of the usual value, the number of rare rows, their most common class and its share.
    """
    shares = X_train[col].value_counts(normalize=True)  # largest share first
    rare = y_train[X_train[col] != shares.index[0]].value_counts(normalize=True)  # classes of the rare rows
    return {
        "dominant": shares.iloc[0],
        "rare_rows": int((X_train[col] != shares.index[0]).sum()),
        "top_class": rare.index[0] if len(rare) else "-",
        "top_share": rare.iloc[0] if len(rare) else 0.0,
    }


def purity(*columns, target=None):
    """Share of rows in the most common class of their value bin.

    Args:
        *columns: One column, or two for a pair (Series aligned with the target).
        target: Labels to score against; the training labels when omitted (a cross-validation fold passes its own).

    Returns:
        Near the largest class share when the values say little about the class, 1 when they give it.
    """
    target = y_train if target is None else target
    side = round(PURITY_BINS ** (1 / len(columns)))  # 100 bins for one column, 10 x 10 for a pair
    # Equal-count bins; tied values are split in row order, which the shuffled split makes random
    bins = [pd.qcut(c.rank(method="first"), side, labels=False) for c in columns]
    counts = pd.crosstab(bins, target)  # rows per bin and class
    return counts.max(axis=1).sum() / len(target)  # rows in the top class of their bin, as a share


def class_spread(col):
    """Spread of a feature inside each class: the class IQR divided by the class median.

    Args:
        col: Name of a feature in df (raw values).

    Returns:
        One relative spread per class (classes with a median of 0 are left out).
    """
    by_label = df.loc[X_train.index, col].groupby(y_train)  # raw training values, per class
    class_iqr = by_label.quantile(0.75) - by_label.quantile(0.25)
    # A median of 0 becomes NaN, so those classes are dropped instead of dividing by zero
    return (class_iqr / by_label.median().replace(0, np.nan)).dropna()


def is_flood(labels):
    """Mark the flood attacks among class names (every class whose name contains "flood").

    Args:
        labels: Class names (a Series or an Index).

    Returns:
        A boolean array, True for the flood classes.
    """
    return np.asarray(labels.str.contains("flood", case=False))


def batch_stand_ins(rows, iat):
    """Find the features that hold one fixed value inside every recording batch and a different one between batches.

    Args:
        rows: Feature values of training rows.
        iat: Raw inter-arrival time of the same rows; rounded to BATCH_STEP it gives each row's recording batch.

    Returns:
        A tuple (stand_ins, usual, share, batch): the stand-in features, the usual value of every feature in each
        batch, the share of that value in its batch, and the batch of each row (batches with 1% of the rows).
    """
    batch = (iat / BATCH_STEP).round()
    batch = batch[batch.map(batch.value_counts(normalize=True)) >= 1 - NEAR_CONST]  # batches need 1% of the rows
    in_batch = rows.loc[batch.index].groupby(batch)
    usual = in_batch.agg(lambda s: s.value_counts().index[0])  # most common value per batch and feature
    share = in_batch.agg(lambda s: s.value_counts(normalize=True).iloc[0])  # its share of the batch
    stand_in = (share >= NEAR_CONST).all() & (usual.nunique() > 1)  # fixed in every batch, changes between them
    return stand_in.index[stand_in].to_list(), usual, share, batch


def recording_shift(rows, batch, labels):
    """Compare each feature's class medians between the two smaller recording batches.

    Args:
        rows: Feature values of training rows.
        batch: Recording batch of each row (from batch_stand_ins).
        labels: Class of each row.

    Returns:
        A tuple (table, shared): one row per continuous feature with the common factor (median of the class-median
        ratios), its spread over the classes (std / mean) and whether the feature follows the recording; and the
        classes recorded in both batches.
    """
    pair = batch[batch.isin(batch.value_counts().index[1:3])]  # rows of the two batches after the main one
    counts = pd.crosstab(labels[pair.index], pair)
    shared = counts.index[(counts >= MIN_RARE).all(axis=1)]  # classes with real rows in both batches
    in_both = pair.index[labels[pair.index].isin(shared)]
    cont = [c for c in rows if rows[c].nunique() > 2]  # a 0/1 flag has no meaningful ratio
    medians = rows.loc[in_both, cont].groupby([labels[in_both], pair[in_both]]).median()
    first, second = sorted(pair.unique())
    # One ratio per class (a zero median in the first batch gives no ratio)
    ratio = medians.xs(second, level=1) / medians.xs(first, level=1).replace(0, np.nan)
    table = pd.DataFrame({"factor": ratio.median(), "spread": ratio.std() / ratio.mean()})
    table["follows recording"] = (table.spread <= SHIFT_SPREAD) & (np.log(table.factor).abs() >= SHIFT_MIN)
    return table.sort_values("spread"), shared


def hidden_stand_ins(rows, stand_ins):
    """Correlate every feature with another feature times a stand-in, and with that feature alone.

    Args:
        rows: Values of the kept training features.
        stand_ins: Values of the dropped stand-ins for the same rows.

    Returns:
        A DataFrame with one row per (feature, other feature, stand-in): both correlations and whether the feature
        follows the product (correlation >= HIDDEN_CORR) but not the other feature alone (it carries the stand-in).
    """
    cont = [c for c in rows if rows[c].nunique() > 2]
    found = []
    for s in stand_ins:
        for g in cont:
            with_s = rows[cont].corrwith(rows[g] * stand_ins[s])  # every feature against g x stand-in
            alone = rows[cont].corrwith(rows[g])  # every feature against g alone
            found += [{"feature": f, "follows": f"{g} x {s}", "corr with product": with_s[f],
                       "corr without stand-in": alone[f]} for f in cont if f != g]
    table = pd.DataFrame(found)
    table["carries stand-in"] = (table["corr with product"] >= HIDDEN_CORR) & (table["corr without stand-in"] < HIDDEN_CORR)
    return table.sort_values("corr with product", ascending=False)


# ======== Before ========
# ==================================================================
# x.1 Constant and near-constant features
# A feature that holds one value in almost every row tells the classes apart only if its few other values
# belong mostly to one class; otherwise it adds nothing and is dropped.
# ==================================================================
assert LABEL_COL not in X_train, "Target must never be an input feature"
pre_columns = X_train.columns.to_list()  # the columns step x starts from (used again by the fold check in step xv)
dominant = X_train.apply(lambda s: s.value_counts(normalize=True).iloc[0])  # share of the most common value
below = dominant[dominant < NEAR_CONST]  # features that are not reviewed
suspects = dominant.index[dominant >= NEAR_CONST]
const_report = pd.DataFrame([rare_profile(c) for c in suspects], index=suspects)
keep = (const_report.rare_rows >= MIN_RARE) & (const_report.top_share >= MIN_SHARE)  # enough rare rows, mostly one class
const_report["action"] = np.where(keep, "keep", "drop")
print("======== Before ========")
print(f"Features: {X_train.shape[1]} | features whose most common value fills at least {NEAR_CONST:.0%} of the rows: "
      f"{len(suspects)}")
print(f"Table: the review of those features. A feature is kept only if at least {MIN_RARE} rows hold another value "
      f"and at least {MIN_SHARE:.0%} of them belong to one class.")
display(const_report.rename(columns={"dominant": "share of usual value", "rare_rows": "rows with another value",
                                     "top_class": "class of most of them", "top_share": "that class's share"}).round(4))

# ======== Cleaning ========
X_train = drop(const_report.index[~keep], "constant, or near-constant without a class-specific rare value")

# ======== After ========
# How far the decisions are from the thresholds
enough_rare = const_report[~keep & (const_report.rare_rows >= MIN_RARE)]  # dropped by the class-share rule
one_class = const_report.rare_rows[const_report.top_share >= MIN_SHARE]  # rare rows of features passing the share rule
min_rare_low = int(max(one_class[one_class < MIN_RARE], default=0))  # largest count below MIN_RARE
min_rare_high = int(min(one_class[one_class >= MIN_RARE], default=0))  # smallest count at or above it
print("\n======== After ========")
print(f"Dropped: {int((~keep).sum())} ({', '.join(const_report.index[~keep])}) | kept: {', '.join(const_report.index[keep])} "
      f"| features left: {X_train.shape[1]}")
print(f"Margins: the most uniform feature not reviewed is {below.idxmax()} ({below.max():.2%} one value); the highest class "
      f"share among the dropped features with at least {MIN_RARE} rare rows is {enough_rare.top_share.max():.1%}; any MIN_RARE from {min_rare_low + 1} to "
      f"{min_rare_high} gives the same decisions")

# ======== Conclusion ========
print("\n======== Conclusion ========")
kept_near = const_report[keep]  # near-constant features kept by the review
reasons = [f"{f} (its rare value marks {row.top_class})" for f, row in kept_near.iterrows()]
print("Only features that cannot help tell the attacks apart were removed; kept:", ", ".join(reasons) or "none")

In [ ]:
# ==================================================================
# x.2 Leakage screen
# A feature that gives the attack class away on its own probably records the capture (when and how the traffic was
# recorded), not the attack. Each feature is scored by its value purity: the training rows are cut into
# PURITY_BINS equal-size value ranges, and purity is the share of rows that belong to the most common class of their
# range (near the largest class share = no information, 1 = the value gives the class away).
# ==================================================================

# ======== Before ========
solo = pd.Series({c: purity(X_train[c]) for c in X_train}).sort_values(ascending=False)  # each feature on its own
leaky = solo.index[solo >= LEAK_PURITY].to_list()
solo_baseline = y_train.value_counts(normalize=True).max()  # always guessing the largest class
noise = pd.Series(np.random.default_rng(SEED).random(len(y_train)), index=y_train.index)  # a column with no information
noise_purity = purity(noise)
print("======== Before ========")
print(f"Limit for a leakage suspect: purity {LEAK_PURITY} | reference points: always guessing the largest class "
      f"{solo_baseline:.3f}, a column of random numbers {noise_purity:.3f}")
print("Highest purity among the features:", ", ".join(f"{c} {v:.3f}" for c, v in solo.head(5).items()))

# A suspect that barely varies inside each class suggests a capture value rather than attack behaviour
leak_spread = {c: class_spread(c) for c in leaky}
other_spread = pd.Series({c: class_spread(c).median() for c in CONT if c not in leaky})  # the other continuous features

# The timing features removed after the split (step viii), scored the same way from their raw training values
raw_session = df.loc[X_train.index, SESSION_FEATURES]
raw_session = raw_session.fillna(raw_session.median())  # median-filled, as in step ix
session_purity = pd.Series({f: purity(raw_session[f]) for f in SESSION_FEATURES})
session_spread = {f: class_spread(f) for f in SESSION_FEATURES}  # relative spread inside each class
print(f"Timing features: iat removed in step viii, flow_duration kept (spread = interquartile range / median inside a class; "
      f"the other continuous features have a typical spread of {other_spread.median():.1%}):")
for f in SESSION_FEATURES:
    print(f"  {f}: purity {session_purity[f]:.3f} | typical spread inside a class {session_spread[f].median():.2%} | "
          f"classes with a spread under 1%: {(session_spread[f] < 0.01).sum()} of {len(session_spread[f])}")

# flow_duration is kept: inside a class it varies widely and does not follow the capture time (raw iat)
link = (df.loc[X_train.index, ["flow_duration", "iat"]].groupby(y_train).corr(method="spearman")
        .xs("flow_duration", level=1)["iat"].dropna())  # rank correlation with iat inside each class
flow_spread = class_spread("flow_duration").median()
print(f"  flow_duration (kept): purity {solo['flow_duration']:.3f} | typical spread inside a class {flow_spread:.0%} | "
      f"classes where it follows iat (rank correlation above 0.5): {(link.abs() > 0.5).sum()} of {len(link)}")
iat_class_medians = df.loc[X_train.index, "iat"].groupby(y_train).median() / 1e6  # one median per class, in millions
print(f"  iat class medians: {iat_class_medians.min():.0f} to {iat_class_medians.max():.0f} million")

# ======== Cleaning ========
# 1. Leakage suspects (none is expected once the timing feature is gone)
X_train = drop(leaky, "possible leakage")
# 2. Batch stand-ins: features that hold one fixed value in each recording batch (the raw iat rounded to BATCH_STEP)
proxies, usual, share, batch = batch_stand_ins(X_train, df.loc[X_train.index, "iat"])
proxy_table = usual[proxies]  # the usual value of each stand-in per batch
stand_in_values = X_train[proxies]  # kept for the hidden stand-in check below
X_train = drop(proxies, "recording-batch stand-in")
# 3. Recording shift: in the classes recorded in both small batches, a feature whose class medians all move by one
#    common factor follows the recording settings, not the attack
shift_table, shift_classes = recording_shift(X_train, batch, y_train)
shifted = shift_table.index[shift_table["follows recording"]].to_list()
X_train = drop(shifted, "recording stand-in (same shift in every class)")
# 4. Hidden stand-ins: a feature that equals another feature times a stand-in carries the stand-in along
products = hidden_stand_ins(X_train, stand_in_values)
carriers = products[products["carries stand-in"]].drop_duplicates("feature").set_index("feature")
hidden = carriers.index.to_list()
X_train = drop(hidden, "hidden stand-in (another feature times a stand-in)")

# ======== After ========
print("\n======== After ========")
print(f"Leakage suspects among the kept features: {leaky or 'none'} (best feature {solo.iloc[0]:.3f}, limit {LEAK_PURITY})")
print(f"Recording-batch stand-ins dropped: {proxies or 'none'}")
if proxies:
    print(f"Table: the usual value of each stand-in in each recording batch (rows: batch). Each holds this value in at "
          f"least {share[proxies].min().min():.2%} of its batch, so it tells the batch, not the attack.")
    display(proxy_table.round(2).rename_axis(f"batch (raw iat / {BATCH_STEP:.0e})"))
    closest = share.drop(columns=proxies).min().max()  # the non-stand-in feature closest to the rule
    print(f"Margin: the stand-ins hold their usual value in at least {share[proxies].min().min():.2%} of every batch "
          f"(rule: {NEAR_CONST:.0%}); the closest other feature reaches only {closest:.2%}")
shift_next = shift_table[~shift_table["follows recording"]].iloc[0]  # the feature closest to the shift rule
print(f"\nTable: class medians in the two small recordings ({len(shift_classes)} classes recorded in both): the common factor "
      f"(second / first) and its spread over the classes. A feature is a recording stand-in when every class moves by "
      f"the same factor (spread at most {SHIFT_SPREAD:.0%}) and the factor differs from 1 by at least {SHIFT_MIN:.0%}.")
display(shift_table.head(5).round(3))
print(f"Recording stand-ins dropped: {shifted or 'none'} | closest kept feature: {shift_next.name} "
      f"(spread {shift_next.spread:.3f}, factor {shift_next.factor:.3f})")
if hidden:
    row = carriers.iloc[0]
    print(f"Hidden stand-ins dropped: {hidden} ({carriers.index[0]} follows {row['follows']} with correlation "
          f"{row['corr with product']:.4f}, but the other feature alone only {row['corr without stand-in']:.4f})")
adds = products[(products["corr without stand-in"] < HIDDEN_CORR) & ~products.feature.isin(hidden)]  # where a stand-in adds something
print(f"Margin: the highest such correlation among the kept features is {adds['corr with product'].max():.4f} "
      f"(rule: {HIDDEN_CORR})")

# Which classes sit in which recording batch: the floods form one batch, the small attacks the others
batch_classes = pd.crosstab(y_train.loc[batch.index], batch)
main_batch = batch_classes.sum().idxmax()
flood_rows = is_flood(batch_classes.index)  # flood classes, by name
in_small = (batch_classes.drop(columns=main_batch) >= MIN_RARE).any(axis=1)  # real rows in a small batch
print(f"Batches: {(batch_classes.loc[flood_rows, main_batch] > 0).sum()} of {flood_rows.sum()} flood classes sit in the "
      f"main batch; {in_small.sum()} classes have rows in a small batch (floods among them: {(in_small & flood_rows).sum()})")

# duration is kept: its class medians look like the time-to-live of a packet (64 on Linux and IoT devices)
dur_medians = df.loc[X_train.index, "duration"].groupby(y_train).median().round(1)
dur_flood = is_flood(dur_medians.index)
dur_mode = dur_medians[dur_flood].mode()[0]  # most common class median among the floods
dur_flood_at_mode = int((dur_medians[dur_flood] == dur_mode).sum())
dur_other_above = int((dur_medians[~dur_flood] > dur_mode).sum())
print(f"duration (kept): {dur_flood_at_mode} of {dur_flood.sum()} flood classes have a median of {dur_mode:g}, "
      f"{dur_other_above} of {(~dur_flood).sum()} other classes are higher (up to {dur_medians[~dur_flood].max():g}); purity {solo['duration']:.3f}, far below the limit")

# Pairs: a fingerprint could be split over two features
top = solo.drop(leaky + proxies + shifted + hidden).index[:PAIR_TOP]
pairs = {}
for k, a in enumerate(top):
    for b in top[k + 1 :]:  # every pair once
        pairs[f"{a} + {b}"] = purity(X_train[a], X_train[b])
pairs = pd.Series(pairs).sort_values(ascending=False)
print(f"Pairs of the {PAIR_TOP} strongest features: best {pairs.index[0]} ({pairs.iloc[0]:.3f}) | "
      f"at or above the limit: {pairs.index[pairs >= LEAK_PURITY].to_list() or 'none'}")

# ======== Conclusion ========
print("\n======== Conclusion ========")
print(f"iat gives the class away (purity {session_purity['iat']:.3f}) and barely varies inside a class, so it records the "
      f"capture; flow_duration does neither and is kept. With iat gone, no kept feature or pair comes near the limit; the "
      f"recording stand-ins {', '.join(proxies + shifted + hidden)} are dropped as well.")


In [ ]:
# ==================================================================
# x.3 Rows made identical by the removed features
# Without the removed columns, some flows now have exactly the same inputs. Repeated training rows add nothing,
# and a test row whose input is already in training would test memory instead of learning.
# ==================================================================

# ======== Before ========
X_test = X_test[X_train.columns]  # the same columns as training, chosen without looking at the test set
feature_cols = X_train.columns.to_list()
BINARY = [c for c in BINARY if c in feature_cols]
CONT = [c for c in CONT if c in feature_cols]
split_rows = {"train": len(X_train), "test": len(X_test)}
print("======== Before ========")
print(f"Features kept by step x: {len(feature_cols)} | training rows: {len(X_train)} | test rows: {len(X_test)}")

# ======== Cleaning ========
train_classes = y_train.value_counts()  # rows per class before the removals
DECIMALS = 6  # rows are compared on rounded values, so floating-point noise counts as equal

# Training: one row per input, with the majority label
key = X_train.round(DECIMALS)  # rounded copy, used only to compare rows
group = key.groupby(feature_cols).ngroup()  # one id per distinct input
labels = y_train.groupby(group)
n_labels = labels.transform("nunique")
majority = labels.transform(lambda s: s.mode()[0])  # most common label; a tie goes to the alphabetically first
conflict = group[n_labels > 1]  # rows whose input carries more than one label
conflict_classes = sorted(y_train[n_labels > 1].unique())
majority_rows = group[y_train == majority]  # rows that carry their input's majority label
kept = majority_rows.drop_duplicates().index  # the first of them for each input
# The disagreeing rows and their majority labels, kept for the fold check of step xv
conflict_table = pd.DataFrame({"group": group[n_labels > 1], "label": y_train[n_labels > 1]})
conflict_majority = conflict_table.groupby("group")["label"].agg(lambda s: s.mode()[0])
X_train, y_train, key = X_train.loc[kept], y_train.loc[kept], key.loc[kept]

# Test: drop rows whose input is already in training or repeated within the test set
test_key = X_test.round(DECIMALS)
seen = pd.MultiIndex.from_frame(test_key).isin(pd.MultiIndex.from_frame(key))  # test inputs already in training
repeat = test_key.duplicated().to_numpy() & ~seen
test_repeats = int(repeat.sum())
X_test, y_test = X_test[~seen & ~repeat], y_test[~seen & ~repeat]

# Training rows each class lost, and the largest gap between the train and test class shares
lost_train = train_classes.sub(y_train.value_counts(), fill_value=0).astype(int)
lost_train = lost_train.sort_values(ascending=False)
share_gap = y_train.value_counts(normalize=True).sub(y_test.value_counts(normalize=True), fill_value=0)
final_gap = share_gap.abs().max()

# ======== After ========
print("\n======== After ========")
print(f"Training rows: {split_rows['train']} -> {len(X_train)} | test rows: {split_rows['test']} -> {len(X_test)} "
      f"({seen.sum()} already in training, {test_repeats} repeated inside the test set)")
print(f"Identical inputs with different labels: {conflict.nunique()} groups ({len(conflict)} rows), each kept once with its "
      f"most common label; classes involved: {', '.join(conflict_classes)}")
print(f"Classes that lost training rows: {(lost_train > 0).sum()} | largest losses: "
      + ", ".join(f"{c} {n}" for c, n in lost_train.head(5).items()))
print(f"Largest difference in a class's share between train and test: {final_gap:.4%}")

# ======== Conclusion ========
print("\n======== Conclusion ========")
print(f"Features: {len(dropped) + len(feature_cols)} -> {len(feature_cols)}. Every training input is now unique and no "
      "test input repeats a training input exactly; near-identical inputs remain and are measured in step xv.")


In [ ]:
# ==================================================================
# x.4 Relevance and correlated features (listed for Phase 2, nothing removed here)
# Which kept features carry little class information, which pairs nearly copy each other, and whether
# protocol_type adds anything to the tcp/udp/icmp flags. Removing redundant features is decided in Phase 2.
# ==================================================================
CORR_LIMIT = 0.95  # pairs at or above this absolute Spearman correlation are listed
GRE_PROTOCOL = 47  # IP protocol number of GRE (RFC 1700)

# ======== Before ========
print("======== Before ========")
print(f"Kept features: {X_train.shape[1]} | a column of random numbers has purity {noise_purity:.3f}")

# ======== After ========
# 1. Relevance: kept features whose purity is barely above the random column
relevance = solo.reindex(X_train.columns).sort_values()
near_noise = relevance.index[relevance < noise_purity + 0.01].to_list()
# 2. Redundancy: pairs of features that rise and fall together (Spearman correlation, robust to extreme values)
rho = X_train.corr(method="spearman").abs()
upper = np.triu(np.ones(rho.shape, dtype=bool), k=1)  # upper triangle: each pair once, no feature with itself
strong = rho.where(upper).stack().sort_values(ascending=False)  # one row per pair, strongest first
strong = strong[strong >= CORR_LIMIT]
# 3. protocol_type against the tcp/udp/icmp flags: GRE flows (protocol 47) carry none of the three flags
raw_protocol = df.loc[X_train.index, "protocol_type"]  # protocol numbers in their original units
gre = raw_protocol == GRE_PROTOCOL
no_flag = X_train[["tcp", "udp", "icmp"]].sum(axis=1) == 0  # flows marked as none of the three
print("\n======== After ========")
print(f"Features within 0.01 of the random column (little class information on their own): {', '.join(near_noise) or 'none'}")
print(f"Pairs with an absolute Spearman correlation of at least {CORR_LIMIT}: {len(strong)}")
print("Table: the 10 most correlated pairs (1 = the two features rise and fall together exactly)")
display(strong.round(3).rename("|rho|").head(10).to_frame())
print(f"GRE flows (protocol {GRE_PROTOCOL}): {gre.sum()}, of which {(gre & no_flag).sum()} carry no tcp/udp/icmp flag; "
      f"{(~gre & no_flag).sum()} other flows carry none either, so only protocol_type identifies GRE")

# ======== Conclusion ========
print("\n======== Conclusion ========")
print("Nothing is removed here: the weak and correlated features are passed to Phase 2, and protocol_type stays "
      "because the flags cannot tell GRE apart.")


### xi. Encoding Categorical Variables

- **Problem:** models need numbers; the target is text.
- **Decision:** `LabelEncoder` on the training labels; keep `protocol_type` numeric, since the flags already mark its common codes and it holds averaged values. SMOTE gives synthetic rows only values that real neighbours have (step xiv), and Phase 2 step 4 shows that removing it changes nothing measurable, so its encoding cannot matter.
- **Alternatives:** one-hot encoding the target adds a column per class; one-hot `protocol_type` cannot hold averages.

In [ ]:
# ======== Before ========
# Every feature is already numeric, so the target is the only text column left
non_numeric = X_train.select_dtypes(exclude="number").columns.to_list()  # should be empty
print("======== Before ========")
print(f"Text feature columns: {non_numeric or 'none'} | the target is text, for example: {', '.join(y_train.head(3))}")
# protocol_type: whole numbers are protocol codes, the other values are averages over several packets
print(f"protocol_type values that are whole protocol codes: {(raw_protocol.dropna() % 1 == 0).mean():.1%} "
      "(the rest are averages over several packets, so it stays numeric)")
# The tcp/udp/icmp flags already mark the most common codes (raw training values)
top_codes = raw_protocol.value_counts().head(3).index
flag_share = df.loc[raw_protocol.index, ["tcp", "udp", "icmp"]].groupby(raw_protocol).mean().loc[top_codes]
print("Table: share of rows with each flag set, for the three most common protocol codes "
      "(6 = TCP, 17 = UDP, 1 = ICMP); each code matches its own flag")
display(flag_share.round(3))

# ======== Cleaning ========
# One integer per class, learned from the training labels (the stratified split puts every class in train)
le = LabelEncoder().fit(y_train)

# ======== After ========
print("\n======== After ========")
print(f"Classes encoded: {len(le.classes_)} | first mappings:", dict(enumerate(le.classes_[:4])))
print("Encoded sample:", le.transform(y_train.head(3)))

# ======== Conclusion ========
print("\n======== Conclusion ========")
print("The target is the only text column; it gets one integer per class, and the files keep the class names so the "
      "modelling phase can refit the same encoder.")


### xii. Detecting and Handling Outliers

- **Problem:** extreme values could distort statistics and distance-based models.
- **Decision:** detect them with the IQR rule; keep them, because inside each class most are normal attack behaviour; compress them in step xiii.
- **Alternatives:** deleting flagged rows deletes whole classes; capping flattens real differences between attacks.

In [ ]:
# ======== Function Definitions ========
def box_row(data, title, filename):
    """Draw one boxplot per feature on a symlog axis.

    Args:
        data: DataFrame of the features to plot, one boxplot per column.
        title: Figure title.
        filename: File name passed to finish_fig.

    Returns:
        The finished figure.
    """
    with plt.style.context(STYLE):
        fig, axes = plt.subplots(1, data.shape[1], figsize=WIDE)  # one panel per feature
        # One boxplot per panel, outliers drawn as small red dots
        for ax, col in zip(axes, data):
            ax.boxplot(data[col], flierprops={"markersize": 2, "markeredgecolor": "indianred"})
            ax.set(xticks=[], title=col)
            ax.set_yscale("symlog", linthresh=1)  # symlog shows huge and tiny values together
            ax.title.set_fontsize(9)
        fig.suptitle(title)
        fig.tight_layout()
    return finish_fig(fig, filename)


def iqr_flags(frame):
    """Mark the cells outside the IQR fences of their own column.

    Args:
        frame: DataFrame of continuous features.

    Returns:
        A boolean DataFrame (True = outlier cell) for the columns with an IQR above zero.
    """
    q1, q3 = frame.quantile(0.25), frame.quantile(0.75)
    iqr = q3 - q1
    cols = iqr.index[iqr > 0]  # IQR = 0 gives no usable fences
    # Cells below the lower fence or above the upper fence
    return (frame[cols] < q1[cols] - 1.5 * iqr[cols]) | (frame[cols] > q3[cols] + 1.5 * iqr[cols])


# ======== Before ========
# Fences at Q1 - 1.5 x IQR and Q3 + 1.5 x IQR, once for the whole training set ...
flag = iqr_flags(X_train[CONT])  # True = outlier cell
outlier_pct = flag.mean() * 100  # % of rows flagged, per feature
by_class = flag.any(axis=1).groupby(y_train).mean()  # share of each class's rows flagged
global_checked = flag.size  # cells in the columns with a usable global IQR

# ... and once inside each class: are the extremes normal for their own attack?
class_flags = [iqr_flags(X_train.loc[y_train == c, CONT]) for c in y_train.unique()]
within_rows = pd.concat([f.any(axis=1) for f in class_flags]).reindex(X_train.index)  # flagged by its own class
within_cells = sum(int(f.to_numpy().sum()) for f in class_flags)
within_checked = sum(f.size for f in class_flags)  # cells each class could check (IQR above zero)

# The four features whose range is widest compared with their IQR (plotted in the boxplots below)
iqr = X_train[CONT].quantile(0.75) - X_train[CONT].quantile(0.25)
spread_ratio = (X_train[CONT].max() - X_train[CONT].min()) / iqr.where(iqr > 0)
top = spread_ratio.nlargest(4).index

# Placeholder codes were converted in step v, so none may remain here
codes_left = int(((X_train[CONT] < 0) | (X_train[CONT] >= BIG)).to_numpy().sum())
assert codes_left == 0, "Placeholder codes should have been converted in step v"

print("======== Before ========")
print(f"IQR rule (a value outside Q1 - 1.5 x IQR or Q3 + 1.5 x IQR is an outlier) on {len(CONT)} continuous features: "
      f"{(outlier_pct > 0).sum()} have outliers, {(iqr == 0).sum()} cannot be checked (IQR = 0)")
print(f"Training rows with at least one outlier: {flag.any(axis=1).mean():.1%} | classes with 90% or more of their rows "
      f"flagged: {(by_class >= 0.9).sum()} of {len(by_class)} (every row: {(by_class == 1).sum()})")
print(f"With the fences computed inside each class instead: {within_rows.mean():.1%} of rows and "
      f"{within_cells / within_checked:.1%} of checked cells flagged (global fences: {int(flag.to_numpy().sum()) / global_checked:.1%} "
      "of cells), so most extremes are normal for their own attack")
print(f"Placeholder codes left (removed in step v): {codes_left} | largest value: {X_train[CONT].max().max():.3g}")

# Evidence for keeping them: deleting every flagged row would wipe out whole classes
if_deleted = pd.DataFrame({"Share flagged": by_class, "Training rows": y_train.value_counts()})
# Rows each class would keep if its flagged rows were deleted
if_deleted["Rows left if deleted"] = (if_deleted["Training rows"] * (1 - if_deleted["Share flagged"])).round().astype(int)
print("Table: the classes that would lose most if every flagged row were deleted")
display(if_deleted.sort_values("Share flagged", ascending=False).head(8).round(3))

# ======== Cleaning ========
# Nothing is removed or changed: the extremes are genuine attack behaviour (step xiii compresses them with log1p)

# ======== After ========
print("\n======== After ========")
print(f"Rows kept: {len(X_train)} training, {len(X_test)} test; every extreme value is kept unchanged")

# ======== Conclusion ========
print("\n======== Conclusion ========")
print("Deleting the outliers would delete most of several attack classes, so they are kept; their influence on "
      "distance-based models is reduced by the log transform and the scaling of step xiii.")

# ======== Figures ========
# Plot: share of outliers per feature, then boxplots of the widest features
display(barh(outlier_pct.sort_values(ascending=False).head(15), "Share of IQR outliers per feature (train, most affected)",
             "% of training rows outside 1.5 x IQR", "fig8_iqr_outliers"),
        box_row(X_train[top], "Boxplots of the training features with the widest range relative to their IQR (symlog axis)",
                "fig9_boxplots"))

### xiii. Skew Reduction and Feature Scaling

- **Problem:** features on very different scales, with long tails, would dominate distance-based models.
- **Decision:** `log1p` on features with skewness above 2, then `StandardScaler` on every continuous feature, both fitted on the training part.
- **Alternatives:** min-max scaling is ruined by one extreme flow; robust scaling is unnecessary after `log1p`.

In [ ]:
SKEW_LIM = 2  # features with a skewness above this get a log transform

# ======== Before ========
# Find the strongly right-skewed continuous features (on the training set)
skew = X_train[CONT].skew()  # skewness of each continuous feature
skewed = skew.index[skew > SKEW_LIM].to_list()  # strongly skewed features
skew_before = skew[skewed].abs().mean()  # their average skew, compared below
others = skew.drop(skewed)  # continuous features below the limit
print("======== Before ========")
print(f"Continuous features with a skewness above {SKEW_LIM} (a long tail of large values): {len(skewed)} of {len(CONT)} "
      f"| their mean |skew|: {skew_before:.1f} | lowest: {skew[skewed].min():.2f}"
      + (f" | highest of the others: {others.max():.2f}" if len(others) else ""))
print(f"Feature scales: standard deviations from {X_train[CONT].std().min():.3g} to {X_train[CONT].std().max():.3g}")

# Log1p is only defined for values of 0 and above
no_negatives = all((part[skewed] >= 0).to_numpy().all() for part in (X_train, X_test))
assert no_negatives, "log1p needs non-negative values"

# ======== Cleaning ========
# 1. Compress the long tails with log(1 + x)
for part in (X_train, X_test):
    part[skewed] = np.log1p(part[skewed])

# 2. Standardise the continuous features to mean 0 and std 1 (learned from train only)
scaler = StandardScaler().fit(X_train[CONT])
for part in (X_train, X_test):
    part[CONT] = scaler.transform(part[CONT])

# ======== After ========
skew_after = X_train[skewed].skew().abs().mean()  # the same features after the transform
print("\n======== After ========")
print(f"Mean |skew| of those features: {skew_before:.1f} -> {skew_after:.1f}")
print(f"Scaled training features: largest |mean| {X_train[CONT].mean().abs().max():.1e}, "
      f"standard deviations {X_train[CONT].std().round(3).unique()}")

# ======== Conclusion ========
print("\n======== Conclusion ========")
print("Long tails are compressed instead of removed, and every continuous feature now has the same scale, so no "
      "feature dominates a distance-based model; the scaler was fitted on the training part only.")


### xiv. Addressing Class Imbalance

- **Problem:** strong class imbalance lets a model ignore the small attacks.
- **Decision:** class weights by default; SMOTE (`SMOTENC`) only for models without class weights (KNN), applied inside the training folds with `prepare_fold()`, never to an exported file or the test set; shown here on a training copy.
- **Alternatives:** undersampling loses flood traffic; random oversampling copies rows; plain SMOTE breaks the 0/1 flags.

In [ ]:
MAX_UPSAMPLE = 10  # a small class grows to at most 10 times its original size
K_NEIGHBORS = 5  # each new row lies between a real row and one of its 5 nearest neighbours in the same class


# ======== Function Definitions ========
def smote(X, y, to_units, max_growth=MAX_UPSAMPLE):
    """Create new rows for the classes below the median class size with SMOTENC and keep those that obey the flow rules.

    Args:
        X: Training features (the test set is never resampled).
        y: Series of class labels aligned with X.
        to_units: Function that turns new rows back into original units, so the flow rules can be checked.
        max_growth: Largest factor a small class may grow by; it otherwise grows to the median size.

    Returns:
        A tuple (X, y, breaks): the original rows followed by the new rows that obey the flow rules, and the
        number of new rows removed for breaking one.
    """
    counts = y.value_counts()  # rows per class
    median = counts.median()  # target size for the small classes
    small = counts[counts < median]  # only these classes get new rows
    targets = {c: min(int(median), n * max_growth) for c, n in small.items()}  # final size of each small class

    # SMOTENC: continuous values lie between two real rows; each 0/1 flag takes the most common neighbour value
    flags = [c in BINARY or c == "protocol_type" for c in X.columns]  # True for the 0/1 indicators and protocol_type
    sampler = SMOTENC(categorical_features=flags, sampling_strategy=targets, k_neighbors=K_NEIGHBORS, random_state=SEED)
    X_bal, y_bal = sampler.fit_resample(X, y)

    # The new rows must obey the flow rules of step ix, checked in the original units (SMOTENC lists them last)
    n_real = len(X)
    breaks = rule_breaks(to_units(X_bal.iloc[n_real:].copy()), tol=1e-9).any(axis=1).to_numpy()  # tol: rounding
    keep_rows = np.concatenate([np.ones(n_real, dtype=bool), ~breaks])  # every real row, and the new rows that pass
    return X_bal[keep_rows], y_bal[keep_rows], int(breaks.sum())


def to_train_units(rows):
    """Undo the step xiii scaling and log transform of the training part.

    Args:
        rows: Scaled rows (modified in place).

    Returns:
        The rows in their original units.
    """
    rows[CONT] = scaler.inverse_transform(rows[CONT])  # undo the scaling
    rows[skewed] = np.expm1(rows[skewed])  # undo log1p
    return rows


# ======== Before ========
counts = y_train.value_counts()  # training rows per class
below_median = (counts < counts.median()).sum()  # classes that will get new rows
train_imbalance = counts.max() / counts.min()  # imbalance of the training rows
print("======== Before ========")
print(f"Training rows: {len(y_train)} | largest class / smallest class: {train_imbalance:.1f}:1 | "
      f"classes below the median size ({int(counts.median())} rows): {below_median} of {len(counts)}")

# ======== Cleaning ========
# Method 1 (default for every model that accepts it): balanced class weights, the formula of class_weight="balanced"
class_weights = len(y_train) / (len(counts) * counts)  # one weight per class: rare classes weigh more

# Method 2 (for models without class weights, such as KNN): SMOTE, shown here on a copy of the training rows.
# The copy is not exported: in a shared file the synthetic rows would leak into every cross-validation fold and
# distort the feature analysis and the clustering, so the modelling phase applies SMOTE inside each training
# fold with prepare_fold() (step xv). New rows are kept only if they obey the flow rules of step ix.
assert counts.min() > K_NEIGHBORS, "SMOTE needs more rows in every class than neighbours"
X_train_bal, y_train_bal, smote_rule_breaks = smote(X_train, y_train, to_train_units)

# ======== After ========
flags_valid = X_train_bal[BINARY].isin([0, 1]).all().all()  # SMOTENC must keep every flag at 0 or 1
assert flags_valid, "SMOTE turned a 0/1 flag into another value"
bal_counts = y_train_bal.value_counts()
print("\n======== After ========")
print(f"Class weights: {class_weights.min():.2f} for the largest class ({class_weights.idxmin()}) up to "
      f"{class_weights.max():.2f} for the smallest ({class_weights.idxmax()})")
print(f"SMOTE copy: {len(y_train)} -> {len(y_train_bal)} rows ({len(y_train_bal) - len(y_train)} new, "
      f"{smote_rule_breaks} more removed for breaking a flow rule) | largest / smallest class: "
      f"{bal_counts.max() / bal_counts.min():.1f}:1 | 0/1 flags still 0 or 1: {flags_valid} | "
      f"exact copies: {X_train_bal.duplicated().sum()}")

# ======== Conclusion ========
print("\n======== Conclusion ========")
print("Class weights are the default; SMOTE is applied only inside the training folds of models without class "
      "weights, never to the exported files or the test set.")

# ======== Figures ========
before_after = pd.DataFrame({"Before": counts, "After SMOTE (training copy)": bal_counts}).loc[counts.index]
display(barh(before_after, "Training class counts before vs after SMOTE (log scale)", "Number of rows",
             "fig10_class_balance", logx=True))

### xv. Final Validation and Export

- **Problem:** the exported data must be clean, and later phases need a leakage-free way to cross-validate.
- **Decision:** final assertions, a fold check, a `prepare_fold()` self-check and a near-twin check; export only `cleaned_train.csv` (real rows) and `cleaned_test.csv`, then show the raw-vs-cleaned table.
- **Alternatives:** exporting SMOTE rows leaks into cross-validation; keeping repeated test inputs would test memory instead of learning.

In [ ]:
# ======== Function Definitions ========
def refit(rows):
    """Refit the step ix fill values and the step xiii skew choice on some raw training rows.

    Args:
        rows: Raw continuous training values (missing cells still empty).

    Returns:
        A tuple (medians, skewed_here, logged): the fill values, the features with skew above SKEW_LIM on these rows,
        and the filled rows with log1p on the exported skewed features (for comparing the scaler).
    """
    medians = rows.median()
    filled = rows.fillna(medians)
    skew_here = filled.skew()
    logged = filled.copy()
    logged[skewed] = np.log1p(logged[skewed])
    return medians, set(skew_here.index[skew_here > SKEW_LIM]), logged


def prepare_fold(train_idx, val_idx):
    """Rebuild the cleaned data of one cross-validation fold, fitting every value on the training fold only.

    Args:
        train_idx: Positions of the training-fold rows among the real training rows (X_train).
        val_idx: Positions of the validation-fold rows among the real training rows.

    Returns:
        A tuple (X_tr, y_tr, X_va, y_va): the training fold imputed, transformed, scaled and SMOTE-resampled, and the
        validation fold imputed, transformed and scaled with values learned from the training fold.
    """
    tr, va = raw_kept.iloc[train_idx].copy(), raw_kept.iloc[val_idx].copy()
    y_tr, y_va = y_train.iloc[train_idx], y_train.iloc[val_idx]
    medians, modes = tr[CONT].median(), tr[BINARY].mode().iloc[0]  # step ix fill values
    for part in (tr, va):
        impute(part, medians, modes)
    skew_here = tr[CONT].skew()
    logged = skew_here.index[skew_here > SKEW_LIM]  # step xiii skew choice, on the filled training fold
    for part in (tr, va):
        part[logged] = np.log1p(part[logged])
    fold_scaler = StandardScaler().fit(tr[CONT])  # step xiii scaler
    for part in (tr, va):
        part[CONT] = fold_scaler.transform(part[CONT])

    def to_fold_units(rows):
        """Undo this fold's scaling and log transform, so smote() can check the flow rules of its new rows.

        Args:
            rows: Scaled rows of the training fold (modified in place).

        Returns:
            The rows in their original units.
        """
        rows[CONT] = fold_scaler.inverse_transform(rows[CONT])
        rows[logged] = np.expm1(rows[logged])
        return rows

    X_tr, y_tr, _ = smote(tr, y_tr, to_fold_units)  # step xiv, training fold only
    return X_tr, y_tr, va, y_va


def near_keys(X, decimals=None):
    """Give near-identical rows the same key: a hash of the values rounded to a number of decimals.

    Args:
        X: Scaled feature rows.
        decimals: Decimals kept before hashing (NEAR_DECIMALS when omitted).

    Returns:
        A NumPy array with one key per row.
    """
    decimals = NEAR_DECIMALS if decimals is None else decimals
    return pd.util.hash_pandas_object(X.round(decimals) + 0.0, index=False).to_numpy()  # + 0.0 turns -0.0 into 0.0


# ======== Before ========
print("======== Before ========")
print(f"Final data: {len(X_train)} training rows, {len(X_test)} test rows, "
      f"{X_train.shape[1]} features. Checks: completeness, columns, duplicates, refitting on folds, near-identical inputs.")

# ======== After ========
print("\n======== After ========")
# 1. No missing or infinite values, the same columns everywhere, the target never an input, every test class in train
parts = {"train": X_train, "SMOTE training copy": X_train_bal, "test": X_test}
for name, part in parts.items():
    assert np.isfinite(part.to_numpy()).all(), f"{name} has missing or infinite values"
    assert list(part.columns) == list(X_train.columns), f"{name} has different columns"
assert LABEL_COL not in X_train, "Target leaked into the inputs"
assert set(y_test) <= set(y_train), "A test class is absent from train"

# 2. Cleaning can make different flows identical, so the duplicate checks are repeated on the final data
assert not X_train_bal.duplicated().any(), "Training rows with identical features (duplicates, conflicts or SMOTE copies)"
assert X_test.merge(X_train_bal).empty, "Test rows also appear in training (leakage)"  # real and SMOTE rows
print("1. Complete, same columns everywhere, target never an input: passed")
print("2. No duplicate or label-conflicting training row, no test row repeated in training: passed")

# 3. Fold check: would refitting on each training fold change the values fitted once on the whole training part?
FOLDS = 5  # the number of folds a later cross-validation would use
folds = list(StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED).split(X_train, y_train))
raw_train = df.loc[X_train.index, pre_columns]  # raw training values of the step x input columns
raw_kept = raw_train[feature_cols]  # the kept features (every one exists in the raw file)
ref_median, _, ref_log = refit(raw_train[CONT])  # the values fitted once on the whole training part
fold_rows = []
# The majority labels of step x.3, re-decided with one fifth of the disagreeing rows left out each time
conflict_order = np.random.default_rng(SEED).permutation(len(conflict_table))
conflict_parts = np.array_split(conflict_order, FOLDS)
for fold_no, (fold_idx, _) in enumerate(folds):
    medians, skew_here, logged = refit(raw_train[CONT].iloc[fold_idx])
    fold_y = y_train.iloc[fold_idx]
    # The step x screens, rerun on the training fold with its own labels (median-filled, as in step ix)
    fold_raw = df.loc[fold_y.index, feature_cols + ["iat"]]
    fold_raw = fold_raw.fillna(fold_raw.median())
    dominant_here = raw_train.iloc[fold_idx].apply(lambda col: col.value_counts(normalize=True).iloc[0])
    stand_ins_here = batch_stand_ins(df.loc[fold_y.index, proxies], fold_raw["iat"])[0]
    # The step x.1 class-share rule on this fold: keep a near-constant feature only if its rare rows mark one class
    near_rows = raw_train.iloc[fold_idx][list(suspects)]
    near_rows = near_rows.fillna(near_rows.mode().iloc[0])  # a missing flag takes the usual value, as in step ix
    rare_here = near_rows.ne(near_rows.mode().iloc[0])  # rows that differ from the usual value
    top_here = pd.Series({c: fold_y[rare_here[c].to_numpy()].value_counts(normalize=True).max() if rare_here[c].any()
                          else 0.0 for c in suspects})  # share of the rare rows in their most common class
    keep_here = (rare_here.sum() >= MIN_RARE) & (top_here >= MIN_SHARE)
    majority_here = (conflict_table.drop(conflict_table.index[conflict_parts[fold_no]])
                     .groupby("group")["label"].agg(lambda s: s.mode()[0]))
    fold_rows.append({
        "no kept feature reaches the leakage limit": max(purity(fold_raw[c], target=fold_y) for c in feature_cols) < LEAK_PURITY,
        "iat still above the limit": purity(fold_raw["iat"], target=fold_y) >= LEAK_PURITY,
        "stand-ins still hold": set(stand_ins_here) == set(proxies),
        "near-constant decisions that change": ", ".join(keep.index[keep_here.reindex(keep.index) != keep]) or "none",
        "majority labels that change (groups)": int((majority_here.reindex(conflict_majority.index) != conflict_majority).sum()),
        "same skewed features": skew_here == set(skewed),
        "same near-constant features": set(dominant_here.index[dominant_here >= NEAR_CONST]) == set(suspects),
        # Largest move of a scaler mean (in standard deviations), of a scaler spread and of a fill value (relative)
        "max scaler mean shift (std units)": ((logged.mean() - ref_log.mean()).abs() / ref_log.std()).max(),
        "max scaler std change": (logged.std() / ref_log.std() - 1).abs().max(),
        "max fill value (median) change": ((medians - ref_median).abs() / ref_median.abs().replace(0, np.nan)).max(),
    })
fold_check = pd.DataFrame(fold_rows)
fold_check.index = [f"fold {i}" for i in range(1, FOLDS + 1)]
print(f"3. Table: every fitted value refitted on each of {FOLDS} training folds and compared with the value fitted once "
      "on the whole training part (True = the same decision; the last three columns are the largest changes)")
display(fold_check.round(4))

# Self-check of prepare_fold on the first fold: complete, same columns, validation untouched, scaler from the fold
check_train, check_val = folds[0]
X_tr, y_tr, X_va, y_va = prepare_fold(check_train, check_val)
assert list(X_tr.columns) == list(X_va.columns) == list(X_train.columns), "prepare_fold columns differ"
assert not X_tr.isna().to_numpy().any() and not X_va.isna().to_numpy().any(), "prepare_fold left missing values"
assert len(X_va) == len(check_val), "the validation fold must not be resampled"
assert X_tr.iloc[: len(check_train)][CONT].mean().abs().max() < 1e-6, "the scaler must be fitted on the training fold"
print(f"4. prepare_fold() rebuilds a fold from raw values with SMOTE inside it: training fold {len(check_train)} real + "
      f"{len(X_tr) - len(check_train)} SMOTE rows, validation fold {len(X_va)} real rows: passed")

# 4. Near-identical inputs: test rows (and validation-fold rows) whose rounded input also occurs in training
NEAR_DECIMALS = 3  # scaled values are compared after rounding to this many decimals
train_keys, test_keys = near_keys(X_train), near_keys(X_test)
has_twin = np.isin(test_keys, train_keys)
same_label = pd.MultiIndex.from_arrays([test_keys, y_test]).isin(pd.MultiIndex.from_arrays([train_keys, y_train]))
test_flood = is_flood(y_test)
near_twin = {
    "share": has_twin.mean(),  # test rows with a near twin in training
    "flood": test_flood[has_twin].mean() if has_twin.any() else 0.0,  # share of them in flood classes
    "nonflood": has_twin[~test_flood].mean(),  # non-flood test rows with a near twin
    "same_label": same_label[has_twin].mean() if has_twin.any() else 0.0,  # near twins with the same label
}
print(f"5. Test rows with a near-identical training input (equal after rounding to {NEAR_DECIMALS} decimals): "
      f"{near_twin['share']:.1%}; {near_twin['flood']:.1%} of them are floods, whose repeated packets are the attack itself "
      f"| non-flood test rows with one: {near_twin['nonflood']:.1%} | same label as the twin: {near_twin['same_label']:.1%}")
# Is there a natural rounding level at which near twins become "the same input"? The share at every level
twin_curve = pd.Series({d: np.isin(near_keys(X_test, d), near_keys(X_train, d)).mean() for d in range(6, 0, -1)})
twin_by_class = pd.Series(has_twin).groupby(y_test.to_numpy()).mean()  # share of each class's test rows with a twin
print("   Share of test rows with a training twin at each rounding (decimals: share): "
      + ", ".join(f"{d}: {s:.1%}" for d, s in twin_curve.items())
      + f" | largest class share at {NEAR_DECIMALS} decimals: {twin_by_class.idxmax()} {twin_by_class.max():.1%}")
print("   The share climbs with every coarser rounding, so no level marks where a repeated input ends: exact identity "
      "(step x.3) is the only threshold-free rule. The twins are flood flows whose repetition is the attack itself, and "
      f"{1 - near_twin['same_label']:.1%} of them carry a different label, so they stay in the test set.")
# The same inside cross-validation: each validation fold against its training fold
train_flood = is_flood(y_train)
val_twins = [(np.isin(train_keys[va], train_keys[tr]), train_flood[va]) for tr, va in folds]
near_twin["cv_share"] = float(np.mean([twin.mean() for twin, _ in val_twins]))  # mean over the folds
near_twin["cv_nonflood"] = float(np.max([twin[~flood].mean() for twin, flood in val_twins]))  # the highest fold
print(f"   The same inside cross-validation: {near_twin['cv_share']:.1%} of validation rows (mean of {FOLDS} folds); "
      f"non-flood validation rows with one: at most {near_twin['cv_nonflood']:.2%}")

# ======== Export ========
# The two files the next phases use. Real rows only: SMOTE belongs inside the training folds (step xiv)
X_train.assign(**{LABEL_COL: y_train}).to_csv("cleaned_train.csv", index=False)
X_test.assign(**{LABEL_COL: y_test}).to_csv("cleaned_test.csv", index=False)

# ======== Conclusion ========
print("\n======== Conclusion ========")
print(f"Every check passed. Exported: cleaned_train.csv ({len(X_train)} rows) and cleaned_test.csv ({len(X_test)} rows); "
      f"features {ddf.shape[1] - 1} -> {X_train.shape[1]} "
      f"({len(BINARY)} 0/1 indicators, {len(CONT)} continuous).")
print("Table: every feature removed during cleaning and why")
with pd.option_context("display.max_colwidth", None):  # show the reasons in full
    display(pd.Series(dropped, name="reason").rename_axis("feature").to_frame())

In [ ]:
# ==================================================================
# xv.2 Raw data against the cleaned data
# The same basic quality checks on the raw file and on the two exported parts, side by side.
# ==================================================================

# ======== Function Definitions ========
def quality_checks(X, y):
    """Count the basic data-quality problems of one data set.

    Args:
        X: Feature columns.
        y: Class labels aligned with X.

    Returns:
        A list in the row order of the comparison table: rows, feature columns, text-typed feature columns,
        missing cells, duplicate rows and classes.
    """
    text_cols = X.select_dtypes(exclude="number").shape[1]
    return [len(X), X.shape[1], text_cols, int(X.isna().to_numpy().sum()), int(X.duplicated().sum()), y.nunique()]


# ======== Before ========
# The raw column counts the step v and vi findings: text columns, then empty cells plus text tokens and placeholder codes
raw_column = [len(ddf), ddf.shape[1] - 1, len(object_cols_before), int(missing_values_after), int(rows_removed),
              ddf[LABEL_COL].nunique()]

# ======== After ========
comparison = pd.DataFrame(
    {"Raw data": raw_column,
     "Cleaned train": quality_checks(X_train, y_train),
     "Cleaned test": quality_checks(X_test, y_test)},
    index=["Rows", "Feature columns", "Text-typed feature columns",
           "Missing cells (incl. tokens and codes)", "Duplicate rows", "Classes"],
)
print("======== After ========")
print("Table: the raw file against the exported files (the train/test split, the removed features and the removed "
      "rows explain the differences in size)")
display(comparison)

# ======== Conclusion ========
print("\n======== Conclusion ========")
print("Both exported parts are numeric, complete and free of duplicates, and both hold every class.")


### xvi. Cleaning Iterations: The Data After Each Stage

- **Problem:** show that each cleaning stage fixed what it targeted.
- **Decision:** measure the data (missing cells, duplicates, skewness, scale spread, imbalance) after four stages and assert that each stage improves its target.
- **Alternatives:** fitting a model per stage measures the model, not the data.

In [ ]:
# ======== Function Definitions ========
def quality(X, y):
    """Data-quality numbers of one cleaning iteration.

    Args:
        X: Features of the iteration (raw text columns allowed).
        y: Class labels aligned with X.

    Returns:
        A dict of rows, features, text columns, missing cells, placeholder codes, duplicate rows,
        near-constant features, mean |skew|, scale spread and imbalance ratio.
    """
    num = X.apply(pd.to_numeric, errors="coerce")  # text tokens become NaN, so they count as missing
    cont = [c for c in num if num[c].nunique() > 2]  # skew and scale only mean something for continuous columns
    std = num[cont].std()  # spread of each continuous column
    sizes = y.value_counts()  # rows per class
    top_share = num.apply(lambda s: s.value_counts(normalize=True).iloc[0])  # share of the most common value
    return {
        "Rows": len(X),
        "Features": X.shape[1],
        "Text columns": int((X.dtypes == object).sum()),
        "Missing cells": int((num.isna() | num.isin(placeholders.index)).to_numpy().sum()),  # blanks, text tokens and codes, as in step vii
        "Placeholder codes": int(num.isin(placeholders.index).to_numpy().sum()),  # the exact codes step v found
        "Duplicate rows": int(pd.concat([X, y], axis=1).duplicated().sum()),
        "Near-constant features": int((top_share >= NEAR_CONST).sum()),
        "Mean |skew|": num[cont].skew().abs().mean(),
        "Scale spread": std.max() / std[std > 0].min(),
        "Imbalance ratio": sizes.max() / sizes.min(),
    }


# ======== Before ========
# Each iteration: (what it fixed, its features, its labels)
raw = ddf.set_axis(df.columns, axis=1)  # raw values under the cleaned column names
ITERATIONS = {
    "1. Raw file": ("As delivered", raw.drop(columns=LABEL_COL), raw[LABEL_COL]),
    "2. Data quality": ("Steps iv-ix: names, labels, types, codes, duplicates, empty rows, split, imputation", *stage_ix),
    "3. Features and scale": (
        "Steps x-xiii: constant and leaking features, repeated rows, encoding, outlier review, log1p, scaling",
        X_train,
        y_train,
    ),
    "4. Class imbalance": ("Step xiv: class weights and a SMOTE training copy", X_train_bal, y_train_bal),
}
# One column per iteration: quality() of its features X and labels y (the fix description is not needed here)
table = pd.DataFrame({name: quality(X, y) for name, (_, X, y) in ITERATIONS.items()})  # metrics x iterations
print("======== Before ========")
print("The data is measured at four stages of the cleaning:")
for name, (fix, _, _) in ITERATIONS.items():
    print(f"  {name}: {fix}")

# Every stage must improve the problem it targets
raw_it, quality_it, scale_it, balance_it = table.columns  # column name of each iteration
assert (table.loc[["Text columns", "Missing cells", "Placeholder codes"], quality_it] == 0).all()
assert table.loc["Duplicate rows", scale_it] == 0  # rows made identical by the removed timing features go in step x.3
assert table.loc["Mean |skew|", scale_it] < table.loc["Mean |skew|", quality_it]
assert table.loc["Scale spread", scale_it] < table.loc["Scale spread", quality_it]
assert table.loc["Near-constant features", scale_it] < table.loc["Near-constant features", quality_it]
assert table.loc["Imbalance ratio", balance_it] < table.loc["Imbalance ratio", scale_it]

# One row per iteration, with what it fixed (the report reads this table)
iterations = table.T.rename_axis("Iteration")
iterations.insert(0, "Fix", [fix for fix, _, _ in ITERATIONS.values()])

# ======== After ========
print("\n======== After ========")
print("Table: data-quality measures (rows) after each cleaning stage (columns). "
      "Scale spread = largest / smallest standard deviation; imbalance ratio = largest / smallest class.")
# The same rounding as Table 2 and the figure labels: counts in full, skew and spread to 3 significant digits
# (scientific notation from 1e4), the imbalance ratio to one decimal
shown = table.astype(object)
for metric in table.index:
    if metric == "Imbalance ratio":
        shown.loc[metric] = [f"{v:.1f}" for v in table.loc[metric]]
    elif metric in ("Mean |skew|", "Scale spread"):
        shown.loc[metric] = [f"{v:.1e}" if v >= 1e4 else f"{v:.3g}" for v in table.loc[metric]]
    else:
        shown.loc[metric] = [f"{v:,.0f}" for v in table.loc[metric]]  # counts
display(shown)

# ======== Conclusion ========
print("\n======== Conclusion ========")
print("Each stage removes the problem it targets (checked by the asserts above): stage 2 the missing values and text, "
      "stage 3 the skew, the scale gaps and the near-constant features, stage 4 most of the class imbalance.")

# ======== Figures ========
# One panel per problem, in plain words; the bar of the stage that fixes it is green. (panel title, log axis, stage)
PANELS = {"Missing cells": ("Missing cells", False, quality_it),
          "Mean |skew|": ("Average skewness (how lopsided the values are)", False, scale_it),
          "Scale spread": ("Largest / smallest feature scale (log axis)", True, scale_it),
          "Imbalance ratio": ("Largest / smallest class size", False, balance_it)}
STAGE_LABELS = {raw_it: "Raw\nfile", quality_it: "Data-quality\nfixes", scale_it: "Feature and\nscale fixes",
                balance_it: "SMOTE\n(training copy)"}  # x-axis names, on two lines
stages = [STAGE_LABELS[name] for name in table.columns]
with plt.style.context(STYLE):
    fig, axes = plt.subplots(2, 2, figsize=(10, 5.6))
    for ax, (metric, (title, log, target)) in zip(axes.flat, PANELS.items()):
        values = table.loc[metric]
        colors = ["seagreen" if name == target else "#b0c4de" for name in table.columns]  # the fixing stage stands out
        ax.bar(stages, values, color=colors, width=0.6)
        fmt = "{:.1f}" if metric == "Imbalance ratio" else "{:.3g}"  # the imbalance ratio keeps one decimal, as in Table 2
        labels = [f"{v:.1e}" if v >= 1e4 else fmt.format(v) for v in values]  # large values in scientific notation
        ax.bar_label(ax.containers[0], labels=labels, padding=2, fontsize=7)
        ax.set_title(title, fontsize=9)
        if log:
            ax.set_yscale("log")
            ax.set_ylim(top=values.max() * 100)  # headroom for the labels on a log axis
        else:
            ax.set_ylim(0, values.max() * 1.25)  # headroom for the labels
        ax.tick_params(axis="x", labelsize=7)
        for label, name in zip(ax.get_xticklabels(), table.columns):  # the fixing stage's name in green, bold
            if name == target:
                label.set_color("seagreen")
                label.set_fontweight("bold")
        ax.grid(axis="y", linestyle="--", alpha=0.2)
        ax.set_axisbelow(True)
    fig.suptitle("Data quality after each cleaning stage (lower is better; green = the stage that fixes it)")
    fig.tight_layout()
display(finish_fig(fig, "fig_iterations"))


## Phase 2: Feature Analysis and Selection

- **Problem:** which features matter, which are redundant or leaking, and does a smaller subset keep or improve the models?
- **Decision:** training rows only; stratified folds for every comparison; each subset rebuilt inside every training fold; Random Forest (class weights) and KNN (SMOTE inside each training fold) for classification, a decision tree for leakage, K-Means for clustering.
- **Alternatives:** choosing a subset on all rows and then cross-validating it would let the validation folds shape the choice.

### 1. Load and Audit the Training Data

- **Problem:** the analysis must start from the files a teammate receives.
- **Decision:** load `cleaned_train.csv`, audit it, and group the features into the brief's five families.
- **Alternatives:** reusing Phase 1's frames in memory would hide what the files contain.

In [ ]:
# ======== Function Definitions ========
# Libraries for Phase 2 (scikit-learn, plus threadpoolctl to cap the threads)
import time
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import adjusted_rand_score, f1_score, silhouette_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from threadpoolctl import threadpool_limits
from imblearn.over_sampling import SMOTE  # for a feature set without 0/1 columns, where SMOTENC cannot run

FA_FOLDS = 5  # stratified folds for every comparison
PAIR_TOP = 6  # most important features whose pairs the leakage check also scores
VAL_SIZE = 0.2  # validation share of the training rows in step 3
TREES = 100  # trees in every Random Forest
K_NN = 5  # KNN neighbours (odd, team decision)
PERM_REPEATS = 5  # shuffles per feature for permutation importance
SOLO_LIMIT = 0.70  # single-feature macro F1 that would mark a leakage suspect
PAIR_SHOW = 0.80  # correlated pairs listed in step 2
CORR_LIMITS = (0.95, 0.90, 0.80)  # correlation filters compared in step 4
TOP_K = (10, 15, 20)  # importance cut-offs compared in step 4
K_RANGE = range(2, 16)  # cluster numbers compared in step 5
SIL_SAMPLE = 5000  # rows behind each silhouette score
FOLD_STARTS = 2  # K-Means starts per fit when clustering is repeated on each training fold
N_PROBES = 5  # random columns that set the noise band of the relevance check
TIME_REPEATS = 3  # timings per fold of the chosen subset and all features (step 4), so the CPU saving has a spread
# Features tested by removal: the two the cleaning phase passed on (handoff note) and flow_duration, kept in step viii
PASSED_ON = ("duration", "protocol_type", "flow_duration")

# Size and spread measures, named in the brief; the other families are recognised from the data in "Before"
VOLUME = ["header_length", "total_sum", "total_size", "min", "max", "avg"]
STATISTICAL = ["std", "variance", "covariance", "magnitude", "radius"]

# Attack groups: the class-name prefix, plus the classes whose name has no group prefix
GROUP_PREFIXES = ("DDoS", "DoS", "Mirai", "Recon")
GROUP_OF = {"VulnerabilityScan": "Recon", "MITM-ArpSpoofing": "Spoofing", "DNS_Spoofing": "Spoofing",
            "DictionaryBruteForce": "Brute force"}  # every other class without a prefix is a web attack


def forest():
    """Create the Random Forest used throughout Phase 2.

    Returns:
        An unfitted RandomForestClassifier with balanced class weights and a fixed seed.
    """
    return RandomForestClassifier(n_estimators=TREES, class_weight="balanced", random_state=SEED, n_jobs=CORES)


def knn():
    """Create the KNN classifier used throughout Phase 2.

    Returns:
        An unfitted KNeighborsClassifier with K_NN neighbours.
    """
    return KNeighborsClassifier(n_neighbors=K_NN, n_jobs=CORES)


def balance(X, y):
    """SMOTE for the training rows of a model without class weights, with the class targets of step xiv.

    Args:
        X: Prepared training rows (a training fold, never a validation fold or the test set).
        y: Their class labels.

    Returns:
        A tuple (X, y): the real rows followed by the new rows that obey the flow rules.
    """
    counts = y.value_counts()
    median = counts.median()
    targets = {c: min(int(median), n * MAX_UPSAMPLE) for c, n in counts[counts < median].items()}
    flags = [set(np.unique(X[c])) <= {0, 1} or c == "protocol_type" for c in X.columns]  # as in step xiv
    if any(flags):
        sampler = SMOTENC(categorical_features=flags, sampling_strategy=targets, k_neighbors=K_NEIGHBORS, random_state=SEED)
    else:
        sampler = SMOTE(sampling_strategy=targets, k_neighbors=K_NEIGHBORS, random_state=SEED)
    # SMOTE reads only the rows of the classes it grows (their neighbours and their spread), so the other rows are left
    # out of the call: exactly the same new rows, about five times faster
    growing = y.isin(list(targets)).to_numpy()
    X_grown, y_grown = sampler.fit_resample(X[growing], y[growing])
    X_bal = pd.concat([X, X_grown.iloc[growing.sum():]], ignore_index=True)  # the real rows, then the new rows
    y_bal = pd.concat([y, y_grown.iloc[growing.sum():]], ignore_index=True)
    # The flow rules a new row can break read only the 0/1 columns (step xiv), so they are checked on these values
    new = X_bal.iloc[len(X):]
    bad = pd.Series(False, index=new.index)
    protocols = [c for c in ("tcp", "udp", "icmp") if c in new]
    if protocols:
        bad |= new[protocols].sum(axis=1) > 1  # more than one protocol
    tcp_flags = [c for c in new if c.endswith("_flag_number")]
    not_tcp = [c for c in ("udp", "icmp") if c in new]
    if tcp_flags and not_tcp:
        bad |= new[not_tcp].eq(1).any(axis=1) & new[tcp_flags].eq(1).any(axis=1)  # a TCP flag on a UDP or ICMP flow
    keep_rows = np.concatenate([np.ones(len(X), dtype=bool), ~bad.to_numpy()])
    return X_bal[keep_rows], y_bal[keep_rows]


def fit_model(make, X, y):
    """Fit one model on training rows; KNN, which takes no class weights, gets SMOTE inside these rows first.

    Args:
        make: Function that returns an unfitted model (forest or knn).
        X: Training features.
        y: Training labels.

    Returns:
        The fitted model.
    """
    if make is knn:
        X, y = balance(X, y)
    return make().fit(X, y)


MODELS = {"Random Forest": forest, "KNN": knn}  # a tree model and a distance-based model
KEY = {"Random Forest": "random_forest", "KNN": "knn"}  # model name as it appears in feature_summary.csv


def tree_f1(X):
    """Cross-validate a decision tree that sees only a few columns (the leakage check).

    Args:
        X: The columns to score, with rows aligned with y_fa.

    Returns:
        The mean macro F1 over the stratified folds of step 3.
    """
    tree = DecisionTreeClassifier(class_weight="balanced", random_state=SEED)
    return cross_val_score(tree, X, y_fa, cv=folds, scoring="f1_macro").mean()


def fit_score(make, X_tr, y_tr, X_va, y_va):
    """Fit one model on a training fold and score it on the validation fold.

    Args:
        make: Function that returns an unfitted model (forest or knn).
        X_tr: Training-fold features.
        y_tr: Training-fold labels.
        X_va: Validation-fold features.
        y_va: Validation-fold labels.

    Returns:
        A tuple (scores, model): a dict with the macro F1, the F1 of every class, the fit and predict CPU time
        (process time, which other programs on the machine inflate less than clock time) and, for a forest, its
        total tree nodes; and the fitted model.
    """
    start = time.process_time()
    model = fit_model(make, X_tr, y_tr)  # KNN with SMOTE inside the training fold
    fitted_at = time.process_time()
    predicted = model.predict(X_va)
    done = time.process_time()
    is_forest = hasattr(model, "estimators_")
    return {"macro_f1": f1_score(y_va, predicted, average="macro"),
            "class_f1": pd.Series(f1_score(y_va, predicted, average=None, labels=CLASSES), index=CLASSES),
            "seconds": done - start, "fit_seconds": fitted_at - start,
            "predict_ms_per_1000": (done - fitted_at) / len(X_va) * 1e6,  # s per row x 1e6 = ms per 1000 rows
            "nodes": sum(t.tree_.node_count for t in model.estimators_) if is_forest else np.nan}, model


def cluster_scores(X, y, k, starts):
    """Run K-Means and compare its clusters with the attack classes (the labels are used after clustering only).

    Args:
        X: Features to cluster.
        y: Attack classes of the same rows.
        k: Number of clusters.
        starts: K-Means starts (n_init).

    Returns:
        A tuple (scores, km): the adjusted Rand index and the purity (rows in the most common class of their
        cluster), and the fitted K-Means.
    """
    km = KMeans(n_clusters=k, n_init=starts, random_state=SEED).fit(X)
    counts = pd.crosstab(km.labels_, y.to_numpy())  # rows per cluster and class
    return {"ari": adjusted_rand_score(y, km.labels_), "purity": counts.max(axis=1).sum() / len(y)}, km


# ======== Before ========
# Phase 2 starts from the exported files, as a teammate would
train_file = pd.read_csv("cleaned_train.csv")  # real training rows only (SMOTE is applied inside folds, step xiv)
assert len(train_file) == len(X_train), "cleaned_train.csv must hold the real rows only"
X_fa = train_file.drop(columns=LABEL_COL)
y_fa = train_file[LABEL_COL]
CLASSES = sorted(y_fa.unique())  # fixed class order for the per-class F1
FA_FLAGS = [c for c in X_fa if set(X_fa[c].unique()) <= {0, 1}]  # 0/1 indicators
FA_CONT = [c for c in X_fa if c not in FA_FLAGS]  # standardised continuous features

# The five feature families the brief names: TCP flags and protocol indicators are found from the column names and
# the 0/1 values, the size and spread measures are listed above, and the remaining features describe the flow
tcp_family = [c for c in X_fa if c.endswith(("_flag_number", "_count"))]  # flag indicators and flag counts
protocol_family = ["protocol_type"] + [c for c in FA_FLAGS if c not in tcp_family]  # protocol code and indicators
FAMILIES = {"Flow": [c for c in X_fa if c not in tcp_family + protocol_family + VOLUME + STATISTICAL],
            "Protocol": protocol_family, "TCP flags": tcp_family,
            "Volume": [c for c in VOLUME if c in X_fa], "Statistical": [c for c in STATISTICAL if c in X_fa]}  # exported ones
FAMILY_OF = {f: fam for fam, feats in FAMILIES.items() for f in feats}  # feature -> family

# How each feature is measured: the 0/1 indicators and duration (which appears to be the IP time-to-live, step x.2)
# are read from each packet header; every other feature summarises the packets of the record
HEADER_FIELDS = FA_FLAGS + ["duration"]
SOURCE = {f: "packet header field" if f in HEADER_FIELDS else "summary of the record's packets" for f in FAMILY_OF}
AVAILABLE_WHEN = {f: "with every packet" if f in HEADER_FIELDS else "when the record is written" for f in FAMILY_OF}

# ======== After: audit ========
removed = list(dropped)  # features removed in cleaning (step x), with their reasons in dropped
# The real rows must be complete, standardised and free of the features removed in cleaning
assert not X_fa.isna().to_numpy().any(), "the training rows have missing values"
assert (X_fa[FA_CONT].nunique() > 2).all(), "a continuous feature has only two values: a flag is not 0/1"
assert X_fa[FA_CONT].mean().abs().max() < 1e-6, "continuous features are not centred on the real rows"
assert (X_fa[FA_CONT].std() - 1).abs().max() < 1e-3, "continuous features do not have unit variance"
assert not set(removed) & set(X_fa), "a feature removed in cleaning is back"
assert sorted(FAMILY_OF) == sorted(X_fa.columns), "every feature must belong to exactly one family"

removed_reasons = pd.Series(dropped).value_counts()  # features removed per reason
print("======== Before ========")
print(f"Loaded cleaned_train.csv: {len(X_fa)} training rows, {y_fa.nunique()} classes, {X_fa.shape[1]} features "
      f"({len(FA_FLAGS)} 0/1 flags, {len(FA_CONT)} continuous); {len(removed)} features were removed in cleaning:")
for reason, n in removed_reasons.items():
    print(f"  {n} x {reason}")

print("\n======== After ========")
print("Audit passed: no missing values, flags are 0/1, continuous features standardised, no removed feature is back.")
print("Feature families (the brief's five groups):")
for fam, feats in FAMILIES.items():
    print(f"  {fam} ({len(feats)}): {', '.join(feats)}")

print("\n======== Conclusion ========")
print("The data arrives as the cleaning phase exported it, and every feature belongs to exactly one family.")

### 2. Redundancy

- **Problem:** correlated features count the same information twice in distance-based models.
- **Decision:** list the pairs with Spearman correlation; decide which to keep in step 4, by importance.
- **Alternatives:** dropping by column order is arbitrary; PCA hides which features matter.

In [ ]:
# ======== Function Definitions ========
def plot_corr_heatmap(corr, title):
    """Draw a correlation heatmap with the features ordered and separated by family.

    Args:
        corr: Square correlation matrix of the features.
        title: Figure title.

    Returns:
        The figure.
    """
    order = [f for feats in FAMILIES.values() for f in feats]  # features grouped by family
    edges = np.cumsum([len(feats) for feats in FAMILIES.values()])[:-1]  # where each new family starts
    with plt.style.context(STYLE):
        fig, ax = plt.subplots(figsize=(8, 7))
        image = ax.imshow(corr.loc[order, order], cmap="RdBu_r", vmin=-1, vmax=1)  # fixed scale: colours always mean the same
        ax.set_xticks(range(len(order)), order, rotation=90, fontsize=7)
        ax.set_yticks(range(len(order)), order, fontsize=7)
        for edge in edges:  # lines between the families
            ax.axhline(edge - 0.5, color="black", lw=0.8)
            ax.axvline(edge - 0.5, color="black", lw=0.8)
        fig.colorbar(image, ax=ax, shrink=0.8, label="Spearman correlation")
        ax.set_title(title)
        fig.tight_layout()
    return fig


# ======== Before ========
corr_s = X_fa.corr(method="spearman")  # rank correlation: robust to the extreme values kept in step xii
corr_p = X_fa.corr()  # Pearson, for comparison
upper = np.triu(np.ones(corr_s.shape, dtype=bool), k=1)  # upper triangle: each pair once
# Spearman and Pearson side by side, one row per pair of features
pairs = pd.DataFrame({"spearman": corr_s.where(upper).stack(), "pearson": corr_p.where(upper).stack()})
pairs = pairs[pairs["spearman"].abs() >= PAIR_SHOW]  # the strongly related pairs
pairs = pairs.sort_values("spearman", key=abs, ascending=False)  # strongest first, whatever the sign
pairs["gap"] = pairs["pearson"].abs() - pairs["spearman"].abs()  # large: Pearson inflated by extreme rows
pairs["families"] = [f"{FAMILY_OF[a]} / {FAMILY_OF[b]}" for a, b in pairs.index]  # families of the two features
same_family = [FAMILY_OF[a] == FAMILY_OF[b] for a, b in pairs.index]  # pairs inside one family

print("======== Before ========")
print(f"Features: {X_fa.shape[1]} | feature pairs: {int(upper.sum())} | listed: pairs with an absolute Spearman "
      f"correlation of at least {PAIR_SHOW}")

# ======== After ========
widest = pairs["gap"].abs().idxmax()  # the pair where the two measures disagree most
print("\n======== After ========")
print(f"Strongly related pairs: {len(pairs)}, of which {sum(same_family)} inside one family | the two measures "
      f"disagree most for {widest[0]} + {widest[1]} (Pearson minus Spearman {pairs.loc[widest, 'gap']:+.3f})")
print("Table: every strongly related pair, strongest first (gap > 0: extreme values inflate Pearson)")
display(pairs.round(3))

print("\n======== Conclusion ========")
where = "inside" if sum(same_family) > len(pairs) / 2 else "across"
print(f"Most redundancy sits {where} the families; which feature of a pair to keep is decided in step 4, by importance.")

# ======== Figures ========
display(finish_fig(plot_corr_heatmap(corr_s, "Spearman correlation of the training features by family"), "fig_fa_corr"))

### 3. Relevance, Importance and Leakage

- **Problem:** which features and families drive the classification, and does any feature give the class away?
- **Decision:** Gini, permutation and family importance (also per attack group); single-feature and pair trees as the leakage check, also applied to the removed timing feature.
- **Alternatives:** mutual information ignores what the model uses; Gini alone overrates continuous features.

In [ ]:
# ======== Function Definitions ========
def attack_group(label):
    """Name the attack group of a class.

    Args:
        label: Class name.

    Returns:
        The group name.
    """
    if label in GROUP_OF:  # classes whose name has no group prefix
        return GROUP_OF[label]
    prefix = label.split("-")[0]  # the text before the first hyphen, e.g. DDoS
    return prefix if prefix in GROUP_PREFIXES else "Web"  # any other class is a web attack


def group_f1(y_true, y_pred, groups):
    """Compute the macro F1 over the classes of each attack group.

    Args:
        y_true: True labels.
        y_pred: Predicted labels.
        groups: Series mapping each class to its attack group.

    Returns:
        A Series with one macro F1 per attack group.
    """
    scores = {}
    for g, members in groups.groupby(groups):  # one pass per attack group; members holds its classes
        scores[g] = f1_score(y_true, y_pred, labels=list(members.index), average="macro")
    return pd.Series(scores)


# ======== Before ========
# A stratified validation part of the training rows (the test files stay untouched)
X_fit, X_val, y_fit, y_val = train_test_split(X_fa, y_fa, test_size=VAL_SIZE, stratify=y_fa, random_state=SEED)
rf = forest().fit(X_fit, y_fit)
base_pred = rf.predict(X_val)
base_f1 = f1_score(y_val, base_pred, average="macro")  # validation macro F1 with every feature intact
groups = pd.Series({c: attack_group(c) for c in CLASSES})  # class -> attack group
group_base = group_f1(y_val, base_pred, groups)  # macro F1 inside each group, every feature intact

# ======== Cleaning: importance ========
# 1. Gini importance: impurity reduction per feature, from a forest on all training rows (also ranks the pair check)
gini = pd.Series(forest().fit(X_fa, y_fa).feature_importances_, index=X_fa.columns)

# 2. Permutation importance: the drop in validation macro F1 when one feature is shuffled
perm = permutation_importance(rf, X_val, y_val, scoring="f1_macro", n_repeats=PERM_REPEATS,
                              random_state=SEED, n_jobs=1)  # the forest itself already uses CORES cores
perm_drop = pd.Series(perm.importances_mean, index=X_fa.columns)

# The stratified folds of every cross-validated check from here on
folds = StratifiedKFold(n_splits=FA_FOLDS, shuffle=True, random_state=SEED)

# 3. Family importance on every validation fold: all features of a family are shuffled with the same row order,
#    scored on all classes and inside each attack group
rng = np.random.default_rng(SEED)
fold_family, fold_group = [], []  # one result per fold
for fold_train, fold_val in folds.split(X_fa, y_fa):
    X_tr, y_tr = X_fa.iloc[fold_train], y_fa.iloc[fold_train]
    X_va, y_va = X_fa.iloc[fold_val], y_fa.iloc[fold_val]
    fold_rf = forest().fit(X_tr, y_tr)
    fold_pred = fold_rf.predict(X_va)
    fold_base = f1_score(y_va, fold_pred, average="macro")  # this fold, every feature intact
    fold_group_base = group_f1(y_va, fold_pred, groups)
    family_row, group_rows = {}, {}
    for fam, feats in FAMILIES.items():
        shuffled = X_va.copy()
        row_order = rng.permutation(len(X_va))  # one row order for the whole family
        shuffled[feats] = X_va[feats].to_numpy()[row_order]  # same row order for every column of the family
        predicted = fold_rf.predict(shuffled)
        family_row[fam] = fold_base - f1_score(y_va, predicted, average="macro")
        group_rows[fam] = fold_group_base - group_f1(y_va, predicted, groups)
    fold_family.append(pd.Series(family_row))
    fold_group.append(pd.DataFrame(group_rows).T)  # rows: families, columns: attack groups
family_folds = pd.DataFrame(fold_family)  # one row per fold
family_drop = family_folds.mean().sort_values(ascending=False)  # mean drop over the folds
family_std = family_folds.std()  # its spread over the folds
group_drop = sum(fold_group) / len(fold_group)  # mean drop per family and attack group
group_top = group_drop.idxmax()  # family with the largest mean drop in each group
group_agree = pd.DataFrame([g.idxmax() for g in fold_group]).eq(group_top).sum()  # folds with the same top family

# 4. Leakage check: a decision tree that sees one feature only, then each pair of the most important features
solo_f1 = pd.Series({f: tree_f1(X_fa[[f]]) for f in X_fa}).sort_values(ascending=False)
strongest = gini.sort_values(ascending=False).index[:PAIR_TOP]  # ranked on all training rows
pair_names = [(a, b) for k, a in enumerate(strongest) for b in strongest[k + 1:]]  # every pair once
pair_f1 = pd.Series({f"{a} + {b}": tree_f1(X_fa[[a, b]]) for a, b in pair_names}).sort_values(ascending=False)
solo_hits = solo_f1.index[solo_f1 >= SOLO_LIMIT].to_list()  # single features at or above the leakage limit
pair_hits = pair_f1.index[pair_f1 >= SOLO_LIMIT].to_list()  # pairs at or above it
suspects = solo_hits + pair_hits

# 5. The timing feature removed in step viii, scored the same way from its raw training values
phase1_labels = df.loc[X_train.index, LABEL_COL].to_numpy()  # labels of the Phase 1 training rows, in file order
same_rows = (phase1_labels == y_fa.to_numpy()).all()  # must match the exported labels row by row
assert same_rows, "the Phase 1 training rows do not match the rows of cleaned_train.csv"
raw_timing = df.loc[X_train.index, SESSION_FEATURES]  # not in the exported files
raw_timing = raw_timing.fillna(raw_timing.median())  # the few missing cells take the training median
removed_f1 = pd.Series({f: tree_f1(raw_timing[[f]]) for f in SESSION_FEATURES})

# 6. What the capture feature would do: the step 4 forest on the kept features plus the raw iat (trees need no scaling)
with_iat = X_fa.assign(iat=raw_timing["iat"].to_numpy())
iat_f1 = cross_val_score(forest(), with_iat, y_fa, cv=folds, scoring="f1_macro")  # same folds as step 4

# ======== After ========
rank_agree = gini.rank().corr(perm_drop.rank())  # Spearman correlation of the two rankings
perm_std = pd.Series(perm.importances_std, index=X_fa.columns)  # spread over the shuffles
gini_rank, perm_rank = gini.rank(ascending=False), perm_drop.rank(ascending=False)  # rank 1 = most important
val_rows = y_fa.map(groups).value_counts()  # rows behind each group score (each row validates once)
# One table with every measure per feature
importance = pd.DataFrame({"family": pd.Series(FAMILY_OF), "source": pd.Series(SOURCE),
                           "available_when": pd.Series(AVAILABLE_WHEN), "gini": gini,
                           "permutation_drop": perm_drop, "single_feature_f1": solo_f1})
importance = importance.sort_values("permutation_drop", ascending=False)

print("======== Before ========")
print(f"Forest on a stratified {1 - VAL_SIZE:.0%} of the training rows, checked on the other {VAL_SIZE:.0%}: "
      f"macro F1 {base_f1:.3f} with every feature intact (the reference for the drops below)")

print("\n======== After ========")
print("1. Importance. Largest drops in macro F1 when one feature is shuffled (mean +/- spread over the shuffles):")
for f in perm_drop.sort_values(ascending=False).index[:5]:
    print(f"   {f}: {perm_drop[f]:.3f} +/- {perm_std[f]:.3f}")
print(f"   Gini and permutation rankings agree only in part (Spearman rho {rank_agree:.2f}); ranked at least 5 places "
      f"higher by permutation: {', '.join(perm_rank.index[(gini_rank - perm_rank) >= 5])}; by Gini: "
      f"{', '.join(gini_rank.index[(perm_rank - gini_rank) >= 5])}")
print("2. Family importance: drop in macro F1 when a whole family is shuffled (mean +/- spread over the folds):")
for fam in family_drop.index:
    print(f"   {fam}: {family_drop[fam]:.3f} +/- {family_std[fam]:.3f}")
print("3. Table: drop in each attack group's macro F1 when a family is shuffled (rows: families; larger = more needed)")
display(group_drop.round(3))
print("   Most needed family per group (folds that agree, of " + str(FA_FOLDS) + "; classes and validation rows): "
      + "; ".join(f"{g}: {group_top[g]} ({group_agree[g]}; {(groups == g).sum()} classes, {val_rows.get(g, 0)} rows)"
                  for g in group_drop.columns))
print(f"4. Leakage check (a tree on one feature, then on pairs of the {PAIR_TOP} strongest; limit {SOLO_LIMIT}): "
      f"best feature {solo_f1.index[0]} {solo_f1.iloc[0]:.3f}, best pair {pair_f1.index[0]} {pair_f1.iloc[0]:.3f}, "
      f"suspects: {', '.join(suspects) or 'none'}")
print("   The timing feature removed in step viii, scored the same way from its raw values: "
      + ", ".join(f"{f} {v:.3f}" for f, v in removed_f1.items()))
print(f"   Forest on the kept features plus the raw iat: macro F1 {iat_f1.mean():.3f} +/- {iat_f1.std():.3f} "
      "(step 4 gives the score without it)")
print("5. Table: every measure per feature, most important first (permutation_drop: drop in macro F1 when shuffled; "
      "single_feature_f1: the leakage check)")
display(importance.round(4))

# ======== Conclusion ========
print("\n======== Conclusion ========")
iat_verdict = "reaches it on its own, which supports" if removed_f1["iat"] >= SOLO_LIMIT else "stays below it, which does not support"
print(f"The {family_drop.index[0]} and {family_drop.index[1]} families matter most; "
      f"{'no kept feature or pair reaches' if not suspects else 'some kept features reach'} the leakage limit, and "
      f"iat {iat_verdict} its removal in step viii.")

# ======== Figures ========
# Two panels: the permutation ranking of the strongest features, and the family drops
top_features = importance.head(15).iloc[::-1]  # most important at the top of the chart
with plt.style.context(STYLE):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.4), gridspec_kw={"width_ratios": [3, 2]})
    ax1.barh(top_features.index, top_features["permutation_drop"], color="#4c9be8")
    ax1.set(title="Permutation importance of the strongest features", xlabel="Drop in validation macro F1")
    ax1.tick_params(axis="y", labelsize=7)
    ax2.barh(family_drop.index[::-1], family_drop.iloc[::-1], color="seagreen")
    ax2.set(title="Family importance (family shuffled together)", xlabel="Drop in validation macro F1")
    for ax in (ax1, ax2):
        ax.grid(axis="x", linestyle="--", alpha=0.3)
        ax.set_axisbelow(True)
    fig.tight_layout()
display(finish_fig(fig, "fig_fa_importance"))

# Family importance inside each attack group, one cell per family and group
with plt.style.context(STYLE):
    fig, ax = plt.subplots(figsize=WIDE)
    limit = group_drop.abs().max().max()  # symmetric colour scale around zero
    image = ax.imshow(group_drop, cmap="RdBu_r", vmin=-limit, vmax=limit, aspect="auto")
    ax.set_xticks(range(group_drop.shape[1]), group_drop.columns, fontsize=8)
    ax.set_yticks(range(group_drop.shape[0]), group_drop.index, fontsize=8)
    for (r, c), value in np.ndenumerate(group_drop.to_numpy()):  # the drop written in each cell
        ax.text(c, r, f"{value:.2f}", ha="center", va="center", fontsize=7)
    fig.colorbar(image, ax=ax, label="Drop in group macro F1")
    ax.set_title("Family importance inside each attack group")
    fig.tight_layout()
display(finish_fig(fig, "fig_fa_groups"))

### 4. Effect of Feature Subsets on Classification

- **Problem:** a smaller subset is only better if the macro F1 holds across all classes while the models get cheaper.
- **Decision:** rule-based subsets rebuilt in every fold; choose the smallest within one fold spread of the best, no slower, no leakage suspect; compare it paired, per class and in CPU time.
- **Alternatives:** recursive elimination is costly; a hand-tuned threshold needs tuning; the test set never chooses.

In [ ]:
# ======== Function Definitions ========
def corr_filter(features, corr, limit):
    """Keep each feature whose correlation with every feature kept before it is below the limit.

    Args:
        features: Features in order of importance (the more important feature of a pair is met first).
        corr: Absolute Spearman correlation matrix of the features.
        limit: Correlation limit.

    Returns:
        The list of kept features.
    """
    kept = []
    for f in features:
        if all(corr.loc[f, k] < limit for k in kept):  # below the limit with every feature kept so far
            kept.append(f)
    return kept


def relevant_features(X, y):
    """Find the features that contribute more to prediction than a random column.

    Args:
        X: Training features (one training fold, or all training rows).
        y: Their labels.

    Returns:
        The list of relevant features and a Series with every permutation drop, the random columns included.
    """
    # An inner validation part of these rows only, so an outer validation fold never takes part
    X_in, X_out, y_in, y_out = train_test_split(X, y, test_size=VAL_SIZE, stratify=y, random_state=SEED)
    rng = np.random.default_rng(SEED)
    probes = [f"probe_{i}" for i in range(N_PROBES)]  # random columns: they carry no information
    for name in probes:
        X_in = X_in.assign(**{name: rng.standard_normal(len(X_in))})
        X_out = X_out.assign(**{name: rng.standard_normal(len(X_out))})
    inner = forest().fit(X_in, y_in)
    # Permutation drop of every feature, the random columns included, on the inner validation part
    perm = permutation_importance(inner, X_out, y_out, scoring="f1_macro", n_repeats=PERM_REPEATS,
                                  random_state=SEED, n_jobs=1)  # the forest itself already uses CORES cores
    drops = pd.Series(perm.importances_mean, index=X_in.columns)
    relevant = [f for f in X.columns if drops[f] > drops[probes].max()]  # above the noise band of the probes
    return relevant, drops


def candidate_subsets(X, y):
    """Build every candidate feature subset from one set of training rows.

    Args:
        X: Training features (one training fold, or all training rows).
        y: Their labels.

    Returns:
        A tuple (subsets, drops): a dict mapping each subset name to its list of features, and the permutation
        drop of every feature and random column (relevant_features).
    """
    fitted = forest().fit(X, y)  # a forest on these rows only
    ranking = pd.Series(fitted.feature_importances_, index=X.columns).sort_values(ascending=False)  # most important first
    corr = X.corr(method="spearman").abs()
    subsets = {"All": list(X.columns)}  # the full set, for comparison
    # One greedy correlation filter per limit, on all features
    for limit in CORR_LIMITS:
        subsets[f"Corr {limit:.2f}"] = corr_filter(ranking.index, corr, limit)
    # Irrelevant features removed, alone and before each correlation filter
    relevant, drops = relevant_features(X, y)
    subsets["Relevant"] = [f for f in ranking.index if f in relevant]
    for limit in CORR_LIMITS:
        subsets[f"Relevant + Corr {limit:.2f}"] = corr_filter(subsets["Relevant"], corr, limit)
    # One Gini cut-off per size
    for k in TOP_K:
        subsets[f"Top {k}"] = list(ranking.index[:k])
    for f in PASSED_ON:  # the features the cleaning phase passed on for testing
        subsets[f"All without {f}"] = [c for c in X.columns if c != f]
    return subsets, drops


# ======== Cleaning: cross-validated comparison ========
rows, per_class, fold_lists = [], [], []  # per fold, subset and model; per-class F1; each fold's subsets
for fold_no, (fold_train, fold_val) in enumerate(folds.split(X_fa, y_fa)):
    X_fold, y_fold = X_fa.iloc[fold_train], y_fa.iloc[fold_train]
    X_hold, y_hold = X_fa.iloc[fold_val], y_fa.iloc[fold_val]
    fold_lists.append(candidate_subsets(X_fold, y_fold)[0])  # built on the training fold only
    for name, feats in fold_lists[-1].items():
        for model_name, make in MODELS.items():
            scores, _ = fit_score(make, X_fold[feats], y_fold, X_hold[feats], y_hold)
            per_class.append(scores.pop("class_f1").rename((name, model_name)))  # one F1 per class
            rows.append({"fold": fold_no, "subset": name, "model": model_name, "features": len(feats), **scores})
results = pd.DataFrame(rows)

# ======== After ========
# Mean and spread over the folds, one row per subset and model
cv_table = results.groupby(["subset", "model"]).agg(
    features=("features", "mean"), f1_mean=("macro_f1", "mean"), f1_std=("macro_f1", "std"),
    seconds=("seconds", "mean"), seconds_std=("seconds", "std"),
    fit_seconds=("fit_seconds", "mean"), predict_ms_per_1000=("predict_ms_per_1000", "mean"), nodes=("nodes", "mean"))
f1_mean = cv_table["f1_mean"].unstack("model")
f1_std = cv_table["f1_std"].unstack("model")
seconds = cv_table["seconds"].unstack("model")
seconds_std = cv_table["seconds_std"].unstack("model")
sizes = cv_table["features"].unstack("model").iloc[:, 0]  # mean subset size (the same for both models)

# Selection rule: smallest rule-based subset within one fold std of the best and no slower than all features
rule_based = [s for s in f1_mean.index if not s.startswith(("All without", "Relevant"))]  # the checks are not candidates
best = f1_mean.loc[rule_based].idxmax()  # best subset per model
best_mean = pd.Series({m: f1_mean.loc[best[m], m] for m in MODELS})  # mean macro F1 of the best subset, per model
best_std = pd.Series({m: f1_std.loc[best[m], m] for m in MODELS})  # its fold standard deviation
lowest_ok = best_mean - best_std  # the lowest acceptable mean, per model
close = f1_mean.loc[rule_based].ge(lowest_ok).all(axis=1)  # within the margin for every model
not_slower = seconds.loc[rule_based].le(seconds.loc["All"]).all(axis=1)  # no slower than all features
# Leakage condition: no fold's version of the subset may hold a feature that reaches SOLO_LIMIT on its own
suspect_features = set(solo_hits).union(*(pair.split(" + ") for pair in pair_hits))  # both features of a pair
no_suspect = pd.Series({s: not any(set(lists[s]) & suspect_features for lists in fold_lists) for s in rule_based})
eligible = close & not_slower & no_suspect
iat_best = iat_f1.mean() > f1_mean["Random Forest"].max()  # the set with iat beats every candidate
iat_rejected = removed_f1["iat"] >= SOLO_LIMIT  # and fails the leakage condition
chosen = sizes[eligible[eligible].index].idxmin()  # the smallest eligible subset
shortfall = (lowest_ok - f1_mean.loc[rule_based]).clip(lower=0).max(axis=1)  # how far a subset misses the margin

# Paired comparison: the chosen subset against all features on the same validation folds
paired = results.pivot_table(index="fold", columns=["model", "subset"], values="macro_f1")
fold_gain = pd.DataFrame({m: paired[(m, chosen)] - paired[(m, "All")] for m in MODELS})  # one row per fold
paired_time = results.pivot_table(index="fold", columns=["model", "subset"], values="seconds")
time_gain = pd.DataFrame({m: paired_time[(m, "All")] - paired_time[(m, chosen)] for m in MODELS})  # CPU s saved per fold

# The features inside the noise band: each set built with the relevance filter against the same set without it,
# on the same validation rows (the filter removes that fold's noise-band features)
RELEVANCE_PAIRS = {"Relevant": "All", **{f"Relevant + Corr {l:.2f}": f"Corr {l:.2f}" for l in CORR_LIMITS}}
noise_cost = pd.DataFrame({(m, without): paired[(m, without)] - paired[(m, base)]
                           for without, base in RELEVANCE_PAIRS.items() for m in MODELS})  # one row per fold

# The CPU saving timed again: the chosen subset and all features refitted on every fold, TIME_REPEATS timings in all
repeat_rows = results.loc[results["subset"].isin(["All", chosen]), ["fold", "subset", "model", "seconds"]].assign(repeat=0)
repeat_rows = repeat_rows.to_dict("records")  # the first timing is the comparison above
for repeat in range(1, TIME_REPEATS):
    for fold_no, (fold_train, fold_val) in enumerate(folds.split(X_fa, y_fa)):
        for name in ("All", chosen):
            feats = fold_lists[fold_no][name]
            for model_name, make in MODELS.items():
                scores, _ = fit_score(make, X_fa.iloc[fold_train][feats], y_fa.iloc[fold_train],
                                      X_fa.iloc[fold_val][feats], y_fa.iloc[fold_val])
                repeat_rows.append({"fold": fold_no, "subset": name, "model": model_name,
                                    "seconds": scores["seconds"], "repeat": repeat})
repeat_time = pd.DataFrame(repeat_rows).pivot_table(index=["repeat", "fold"], columns=["model", "subset"], values="seconds")
repeat_saving = pd.DataFrame({m: 1 - repeat_time[(m, chosen)] / repeat_time[(m, "All")] for m in MODELS})  # per timing

# Class by class: the change in F1 with the chosen subset against all features, averaged over the folds
class_f1 = pd.DataFrame(per_class)
class_f1.index = pd.MultiIndex.from_tuples(class_f1.index, names=["subset", "model"])  # two-level index: subset, model
class_mean = class_f1.groupby(level=["subset", "model"]).mean()
class_sizes = y_fa.value_counts()
small_classes = class_sizes.index[class_sizes < class_sizes.median()]  # the smaller half of the classes

# ======== After ========
time_saved = 1 - seconds.loc[chosen] / seconds.loc["All"]  # share of fit and predict time saved, per model
forest_nodes = cv_table.xs("Random Forest", level="model")["nodes"]  # mean total tree nodes per subset
efficiency = cv_table.loc[[chosen, "All"], ["features", "fit_seconds", "predict_ms_per_1000"]]
print("======== Before ========")
print(f"Candidate subsets: {len(rule_based)} rules (checks such as 'All without duration' are scored but not chosen) "
      f"| {FA_FOLDS} stratified folds | models: {', '.join(MODELS)}")

print("\n======== After ========")
print("1. Table: macro F1 (mean and spread over the folds), CPU seconds and forest size of every subset and model")
display(cv_table.round(4))
for m in MODELS:
    print(f"   {m}: best {best[m]} {f1_mean.loc[best[m], m]:.3f} | chosen {chosen} {f1_mean.loc[chosen, m]:.3f} | "
          f"all features {f1_mean.loc['All', m]:.3f} +/- {f1_std.loc['All', m]:.3f}")
print(f"2. Selection rule: within one fold spread of the best for both models, no slower than all features, no "
      f"leakage suspect. Passing the margin: {', '.join(close[close].index)} | no slower: "
      f"{', '.join(not_slower[not_slower].index)} | chosen (smallest that passes all three): {chosen}, "
      f"about {sizes[chosen]:.0f} features")
print(f"3. Chosen against all features on the same validation rows (paired): macro F1 higher in "
      + "; ".join(f"{m} {(fold_gain[m] > 0).sum()} of {FA_FOLDS} folds (mean {fold_gain[m].mean():+.4f} +/- "
                  f"{fold_gain[m].std():.4f})" for m in MODELS))
for m in MODELS:
    change = class_mean.loc[(chosen, m)] - class_mean.loc[("All", m)]  # per class
    print(f"   {m}, class by class: smaller classes {change[small_classes].mean():+.4f}, larger classes "
          f"{change.drop(small_classes).mean():+.4f}; classes better / worse by more than 0.01: "
          f"{(change > 0.01).sum()} / {(change < -0.01).sum()}")
print("4. Table: fit time and prediction time per 1,000 rows, chosen subset against all features (CPU)")
display(efficiency.round(3))
print("   CPU time saved: " + "; ".join(f"{m} {time_saved[m]:.1%} (faster in {(time_gain[m] > 0).sum()} of {FA_FOLDS} folds)"
                                       for m in MODELS)
      + f" | forest nodes relative to all features: {forest_nodes[chosen] / forest_nodes['All']:.3f}")
print(f"   Timed {TIME_REPEATS} times on each fold ({len(repeat_saving)} paired timings): CPU time saved "
      + "; ".join(f"{m} {repeat_saving[m].mean():.1%} +/- {repeat_saving[m].std():.1%} (faster in "
                  f"{(repeat_saving[m] > 0).sum()} of {len(repeat_saving)})" for m in MODELS))
print(f"5. Kept features plus the raw iat: forest macro F1 {iat_f1.mean():.3f} against {f1_mean.loc['All', 'Random Forest']:.3f} "
      f"(highest of all sets: {iat_best}); iat alone scores {removed_f1['iat']:.3f}, at or above the leakage limit "
      f"{SOLO_LIMIT}: {iat_rejected}, so this set is excluded")
print("6. Without the features inside the relevance noise band (removed inside each fold), macro F1 change against "
      "the same set with them, on the same validation rows:")
for without, base in RELEVANCE_PAIRS.items():
    print(f"   {without} against {base}: " + "; ".join(
        f"{m} {noise_cost[(m, without)].mean():+.4f} (lower in {(noise_cost[(m, without)] < 0).sum()} of {FA_FOLDS} folds)"
        for m in MODELS))
for f in PASSED_ON:
    change = f1_mean.loc[f"All without {f}"] - f1_mean.loc["All"]  # per model
    within = change.abs() < f1_std.loc["All"]  # smaller than one fold standard deviation of all features
    print(f"7. Without {f}: macro F1 change " + ", ".join(f"{m} {change[m]:+.4f}" for m in MODELS)
          + f" (within one fold spread: {', '.join(m for m in MODELS if within[m]) or 'neither'})")

# ======== Conclusion ========
print("\n======== Conclusion ========")
top_names = [s for s in rule_based if s.startswith("Top")]
top_worse = bool((f1_mean.loc[top_names] < f1_mean.loc["All"]).all().all())  # every Gini cut-off below all features
print(f"{chosen} keeps the macro F1 of all features within the fold noise with {sizes[chosen]:.0f} of {X_fa.shape[1]} "
      f"features and less CPU time" + ("; cutting features by Gini importance alone loses macro F1." if top_worse else "."))

# ======== Figures ========
subset_order = sizes.sort_values().index  # subsets from smallest to largest
x = np.arange(len(subset_order))
width = 0.4  # bar width for the two models
with plt.style.context(STYLE):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.6))
    for i, m in enumerate(MODELS):  # one bar group per subset, one bar per model
        shift = (i - 0.5) * width  # centre the two bars on the subset tick
        ax1.bar(x + shift, f1_mean.loc[subset_order, m], width, yerr=f1_std.loc[subset_order, m], label=m, capsize=2)
        ax2.bar(x + shift, seconds.loc[subset_order, m], width, label=m)
    ax1.set(title="Cross-validated macro F1 by feature subset", ylabel="Macro F1")
    ax1.set_ylim(max(0, f1_mean.min().min() - 0.1), 1)  # zoom in on the differences
    ax2.set(title="Fit and predict CPU time by feature subset", ylabel="CPU seconds per fold")
    for ax in (ax1, ax2):
        ax.set_xticks(x, subset_order, rotation=45, ha="right", fontsize=7)
        ax.legend(fontsize=7)
        ax.grid(axis="y", linestyle="--", alpha=0.3)
        ax.set_axisbelow(True)
    fig.tight_layout()
display(finish_fig(fig, "fig_fa_subsets"))

### 5. Effect of Feature Subsets on Clustering

- **Problem:** does the subset change the clustering?
- **Decision:** K-Means on three feature sets for every k, on all rows and refitted per fold; adjusted Rand index, purity, silhouette and inertia.
- **Alternatives:** synthetic rows would distort the clusters; a full silhouette is slow.

In [ ]:
# ======== Before ========
final_subsets, final_drops = candidate_subsets(X_fa, y_fa)  # the same rules, applied once to all training rows
selected = final_subsets[chosen]
strictest = f"Corr {min(CORR_LIMITS):.2f}"  # the strictest correlation filter
final_relevant = final_subsets["Relevant"]  # the relevance filter on all training rows (from the same call)
probe_band = final_drops.filter(like="probe_")  # the drops of the random columns: the noise band
set_names = dict.fromkeys(["All", chosen, strictest])  # each set once, in this order
cluster_sets = {name: final_subsets[name] for name in set_names}

# ======== Cleaning: K-Means on each feature set ========
rows = []
with threadpool_limits(limits=CORES):  # K-Means would otherwise use every core
    # 1. All training rows: the curves of the figure
    for name, feats in cluster_sets.items():
        for k in K_RANGE:
            scores, km = cluster_scores(X_fa[feats], y_fa, k, starts=10)
            sil = silhouette_score(X_fa[feats], km.labels_, sample_size=SIL_SAMPLE, random_state=SEED)
            rows.append({"features": name, "k": k, "silhouette": sil, "inertia": km.inertia_, **scores})
    # 2. Refitted on each training fold: how much the label-based comparison varies with the data
    fold_rows = []
    for fold_no, (fold_train, _) in enumerate(folds.split(X_fa, y_fa)):
        X_part, y_part = X_fa.iloc[fold_train], y_fa.iloc[fold_train]
        for name in ("All", chosen):
            for k in K_RANGE:
                scores, _ = cluster_scores(X_part[cluster_sets[name]], y_part, k, starts=FOLD_STARTS)
                fold_rows.append({"fold": fold_no, "features": name, "k": k, **scores})
clusters = pd.DataFrame(rows)
fold_clusters = pd.DataFrame(fold_rows)
first_inertia = clusters.groupby("features")["inertia"].transform("first")  # inertia at the smallest k of each set
clusters["relative_inertia"] = clusters["inertia"] / first_inertia  # elbow curves on one scale

# ======== After ========
best_rows = clusters.groupby("features", sort=False)["silhouette"].idxmax()  # best silhouette per feature set
best_k = clusters.loc[best_rows].set_index("features")
purity_wide = clusters.pivot(index="k", columns="features", values="purity")  # same k, different feature sets
ari_wide = clusters.pivot(index="k", columns="features", values="ari")
sil_wide = clusters.pivot(index="k", columns="features", values="silhouette")
rel_inertia = clusters.groupby("features", sort=False)["relative_inertia"].mean()  # lower: inertia falls faster with k
# Refitted on every training fold: at how many k the chosen subset is ahead in every fold
fold_wide = fold_clusters.pivot_table(index=["k", "fold"], columns="features", values=["ari", "purity"])
ari_fold_gap = (fold_wide[("ari", chosen)] - fold_wide[("ari", "All")]).unstack("fold")  # rows k, columns folds
purity_fold_gap = (fold_wide[("purity", chosen)] - fold_wide[("purity", "All")]).unstack("fold")
ari_every_fold = (ari_fold_gap > 0).all(axis=1)  # ahead in every fold at this k
purity_every_fold = (purity_fold_gap > 0).all(axis=1)
ari_gap_by_fold = ari_fold_gap.mean()  # chosen minus all features, mean over k, one value per fold

print("======== Before ========")
print(f"K-Means on {', '.join(cluster_sets)} for k = {min(K_RANGE)} to {max(K_RANGE)}: on all training rows "
      f"({10} starts per fit) and refitted on each of the {FA_FOLDS} training folds ({FOLD_STARTS} starts per fit). "
      "Scores: adjusted Rand index and purity (agreement with the attack classes, which K-Means never sees), "
      "silhouette (how well separated the clusters are) and inertia (the elbow curve).")

print("\n======== After ========")
print("1. Table: the number of clusters with the best silhouette for each feature set, and its scores")
display(best_k.round(3))
print("2. Mean over the same cluster numbers: "
      + "; ".join(f"{name}: adjusted Rand index {ari_wide[name].mean():.3f}, purity {purity_wide[name].mean():.3f}, "
                  f"relative inertia {rel_inertia[name]:.3f}" for name in ari_wide.columns))
for name in ari_wide.columns.drop("All"):  # each reduced set against all features at the same k
    print(f"   {name} against all features: adjusted Rand index higher at {(ari_wide[name] > ari_wide['All']).sum()}, "
          f"purity at {(purity_wide[name] > purity_wide['All']).sum()}, silhouette at "
          f"{(sil_wide[name] > sil_wide['All']).sum()} of {len(K_RANGE)} cluster numbers")
print(f"3. Refitted on each training fold, {chosen} minus all features (adjusted Rand index, mean over k): "
      + ", ".join(f"{g:+.4f}" for g in ari_gap_by_fold)
      + f" | mean {ari_gap_by_fold.mean():+.4f} +/- {ari_gap_by_fold.std():.4f} | ahead in every fold at "
      f"{ari_every_fold.sum()} of {len(ari_every_fold)} cluster numbers (purity: {purity_every_fold.sum()}) | ahead in "
      f"{(ari_fold_gap > 0).to_numpy().sum()} of {ari_fold_gap.size} fold-and-k comparisons")

# ======== Conclusion ========
print("\n======== Conclusion ========")
lead = "higher" if ari_gap_by_fold.mean() > 0 else "not higher"
consistent = "in every fold" if (ari_gap_by_fold > 0).all() else "not in every fold"
print(f"The clusters of {chosen} agree {lead} with the attack classes than those of all features on average "
      f"({consistent}).")

# ======== Figures ========
with plt.style.context(STYLE):
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=WIDE)
    for name, part in clusters.groupby("features", sort=False):  # one line per feature set
        ax1.plot(part["k"], part["relative_inertia"], marker="o", ms=3, label=name)
        ax2.plot(part["k"], part["silhouette"], marker="o", ms=3, label=name)
        ax3.plot(part["k"], part["purity"], marker="o", ms=3, label=name)
    ax1.set(title="Elbow curve", xlabel="Clusters (k)", ylabel="Inertia / inertia at smallest k")
    ax2.set(title="Silhouette score", xlabel="Clusters (k)", ylabel="Silhouette")
    ax3.set(title="Cluster purity", xlabel="Clusters (k)", ylabel="Purity")
    for ax in (ax1, ax2, ax3):
        ax.set_xticks(list(K_RANGE)[::2])  # whole numbers of clusters
        ax.legend(fontsize=7)
        ax.grid(linestyle="--", alpha=0.3)
    fig.tight_layout()
display(finish_fig(fig, "fig_fa_clusters"))

### 6. The Whole Selection, Stage by Stage

- **Problem:** the exported features are already screened, so steps 4-5 cannot show the effect of the whole selection.
- **Decision:** replay the selection from the raw features, stage by stage, inside every fold; measure macro F1, dependence on capture features, macro F1 on another recording (with a control and a split-half test of the stand-in rules), CPU time and K-Means agreement (with its spread over folds).
- **Alternatives:** the exported features show only the last stage; one big removal hides which step matters.

In [ ]:
# ======== Function Definitions ========
RULE_COLS = ["min", "max", "avg", "tcp", "udp", "icmp"]  # the columns the step ix flow-rule repair reads


def stage_fold(cols, train_idx, val_idx):
    """Rebuild one stage of the selection from the raw training values, fitting every value on the training rows.

    Args:
        cols: Features of the stage.
        train_idx: Positions of the training rows among the real training rows.
        val_idx: Positions of the rows transformed with the training values (may be empty).

    Returns:
        A tuple (X_tr, X_va) holding the stage's features, imputed, log-transformed and scaled.
    """
    needed = cols + [c for c in RULE_COLS if c not in cols]  # the step ix repair also needs these columns
    tr = raw_values.iloc[train_idx][needed].copy()
    va = raw_values.iloc[val_idx][needed].copy()
    parts = [part for part in (tr, va) if len(part)]  # an empty part needs no preparation
    binary = [c for c in needed if set(tr[c].dropna().unique()) <= {0, 1}]  # 0/1 indicators
    cont = [c for c in needed if c not in binary]
    medians, modes = tr[cont].median(), tr[binary].mode().iloc[0]  # step ix fill values, from the training rows
    for part in parts:
        impute(part, medians, modes)
    skew_now = tr[cont].skew()
    logged = skew_now.index[skew_now > SKEW_LIM]  # step xiii skew choice, on the training rows
    for part in parts:
        part[logged] = np.log1p(part[logged])
    scaler = StandardScaler().fit(tr[cont])  # step xiii scaler, on the training rows
    for part in parts:
        part[cont] = scaler.transform(part[cont])
    return tr[cols], va[cols]


# ======== Before ========
raw_cols = [c for c in df.columns if c != LABEL_COL]  # every feature of the raw file
raw_values = df.loc[X_train.index, raw_cols]  # the same rows as X_fa, in the same order (checked in step 3)
reason_of = dict(dropped)  # cleaning removals and their reasons (step x)
assert len(raw_cols) == X_fa.shape[1] + len(reason_of), "every raw feature must be kept or removed with a reason"
# The cleaning stages, in pipeline order: each removes the features whose reason holds the word
REMOVALS = {"Without timing (step viii)": "timing", "Without near-constant (step x)": "constant",
            "Without recording stand-ins (step x.2)": "stand-in"}
KIND = {"Without timing (step viii)": "leakage", "Without near-constant (step x)": "low information",
        "Without recording stand-ins (step x.2)": "leakage"}
stages = {"Raw features": raw_cols}
remaining = raw_cols
for name, word in REMOVALS.items():
    remaining = [c for c in remaining if word not in reason_of.get(c, "")]
    stages[name] = remaining
assert sorted(remaining) == sorted(X_fa.columns), "the cleaning stages must end at the exported features"
capture_cols = [c for c in raw_cols if any(w in reason_of.get(c, "") for w in ("timing", "stand-in"))]  # record the capture
COPIES = f"Without copies ({chosen})"  # the Phase 2 stage
KIND[COPIES] = "redundancy"
stage_names = list(stages) + [COPIES]

# ======== Cleaning: every stage rebuilt and scored inside the step 4 folds ========
chain_rows, chain_class, chain_fold_clusters = [], [], []
for fold_no, (fold_train, fold_val) in enumerate(folds.split(X_fa, y_fa)):
    y_tr, y_va = y_fa.iloc[fold_train], y_fa.iloc[fold_val]
    fold_stages = {**stages, COPIES: fold_lists[fold_no][chosen]}  # the copies removed inside this training fold
    new_order = np.random.default_rng(SEED + fold_no).permutation(len(fold_val))  # same shuffle for every stage of the fold
    for name, cols in fold_stages.items():
        X_tr, X_va = stage_fold(cols, fold_train, fold_val)
        # K-Means refitted on this training fold: how much the cluster validity of each stage varies with the data
        with threadpool_limits(limits=CORES):  # K-Means would otherwise use every core
            for k in K_RANGE:
                scores, _ = cluster_scores(X_tr, y_tr, k, starts=FOLD_STARTS)
                chain_fold_clusters.append({"fold": fold_no, "stage": name, "k": k, **scores})
        # Capture-dependence copy of the validation fold: the capture features shuffled together, so they say nothing
        held = [c for c in cols if c in capture_cols]
        X_new = X_va.copy()
        X_new[held] = X_va[held].to_numpy()[new_order]
        for model_name, make in MODELS.items():
            scores, model = fit_score(make, X_tr, y_tr, X_va, y_va)
            chain_class.append(scores["class_f1"].rename((name, model_name)))  # one F1 per class
            # A stage without capture features scores the same on the new-recording copy
            new_f1 = f1_score(y_va, model.predict(X_new), average="macro") if held else scores["macro_f1"]
            chain_rows.append({"fold": fold_no, "stage": name, "model": model_name, "features": len(cols),
                               "macro_f1": scores["macro_f1"], "new_recording_f1": new_f1, "seconds": scores["seconds"]})

# K-Means on all training rows of each stage (the copies removed by the rule applied to all rows, step 5)
chain_cluster_rows = []
all_rows = np.arange(len(X_fa))
with threadpool_limits(limits=CORES):  # K-Means would otherwise use every core
    for name, cols in {**stages, COPIES: selected}.items():
        X_stage, _ = stage_fold(cols, all_rows, all_rows[:0])
        if name == list(stages)[-1]:  # the rebuilt exported set should match the exported values
            rebuilt_gap = (X_stage - X_fa[X_stage.columns].to_numpy()).abs().to_numpy().max()
        for k in K_RANGE:
            chain_cluster_rows.append({"stage": name, "k": k, **cluster_scores(X_stage, y_fa, k, starts=10)[0]})

# Cross-recording test: the classes recorded in both small batches (step x.2) are trained with one recording and
# tested on the other, all other rows staying in training. A feature that records the capture helps inside one
# recording but not on another, so this is the closest the data comes to new traffic.
recording = (df.loc[X_train.index, "iat"] / BATCH_STEP).round().to_numpy()  # recording batch of each training row
twice = y_fa.isin(shift_classes).to_numpy()  # rows of the classes recorded in both small batches
cross_rows = []
NO_FLOW = "Exported without flow_duration"  # a check of step viii, not a stage of the selection
exported_cols = stages[list(stages)[-1]]
# Control: is the test true by construction? The kept features that shift most between the two recordings (but
# not by one common factor, step x.2) are removed, and each removed stand-in is added back on its own
uneven = shift_table.drop(index=shifted + hidden, errors="ignore").loc[lambda t: t.index.isin(exported_cols)]
uneven = uneven.dropna(subset=["spread"])  # a shift measured over several classes (one class gives no spread)
SHIFTING = np.log(uneven["factor"]).abs().nlargest(3).index.to_list()  # the three largest uneven shifts
NO_SHIFTING = f"Exported without {', '.join(SHIFTING)}"
STAND_INS = proxies + shifted + hidden  # every recording stand-in dropped in step x.2
controls = {NO_SHIFTING: [c for c in exported_cols if c not in SHIFTING],
            **{f"Exported plus {f}": exported_cols + [f] for f in STAND_INS}}
for name, cols in {**stages, COPIES: selected, NO_FLOW: [c for c in exported_cols if c != "flow_duration"],
                   **controls}.items():
    for held in sorted(pd.Series(recording).value_counts().index[1:3]):  # the two small recordings
        test_pos = np.where(twice & (recording == held))[0]
        train_pos = np.where(~(twice & (recording == held)))[0]
        X_tr, X_te = stage_fold(cols, train_pos, test_pos)
        for m, make in MODELS.items():
            predicted = fit_model(make, X_tr, y_fa.iloc[train_pos]).predict(X_te)
            cross_rows.append({"stage": name, "model": m, "held_out": held,
                               "macro_f1": f1_score(y_fa.iloc[test_pos], predicted, average="macro",
                                                    labels=sorted(shift_classes))})
cross_table = pd.DataFrame(cross_rows).pivot_table(index="stage", columns=["model", "held_out"], values="macro_f1")

# ======== After ========
chain = pd.DataFrame(chain_rows)
chain_clusters = pd.DataFrame(chain_cluster_rows)
# One row per stage: size, macro F1 and CPU time of both models, cluster validity over the same k
chain_table = chain.groupby(["stage", "model"]).agg(f1_mean=("macro_f1", "mean"), f1_std=("macro_f1", "std"),
                                                     new_f1=("new_recording_f1", "mean"),
                                                     seconds=("seconds", "mean")).unstack("model")
chain_table.columns = [f"{m} {measure}" for measure, m in chain_table.columns]  # e.g. "KNN f1_mean"
chain_table.insert(0, "features", chain.groupby("stage")["features"].mean())  # mean size over the folds
chain_table.insert(0, "kind", pd.Series(KIND))
chain_table = chain_table.join(chain_clusters.groupby("stage")[["ari", "purity"]].mean())
for m in MODELS:  # cross-recording macro F1, mean of the two directions
    chain_table[f"{m} cross_f1"] = cross_table[m].mean(axis=1)
chain_table = chain_table.loc[stage_names]

# Each removal against the stage before it, on the same validation rows (paired), and class by class
chain_paired = chain.pivot_table(index="fold", columns=["model", "stage"], values="macro_f1")
chain_class = pd.DataFrame(chain_class)
chain_class.index = pd.MultiIndex.from_tuples(chain_class.index, names=["stage", "model"])
chain_class = chain_class.groupby(level=["stage", "model"]).mean()  # mean per-class F1 over the folds
cluster_wide = chain_clusters.pivot(index="k", columns="stage", values="ari")  # same k, different stages
step_rows = []
for before, after in zip(stage_names[:-1], stage_names[1:]):
    for m in MODELS:
        change = chain_paired[(m, after)] - chain_paired[(m, before)]  # one value per fold
        class_change = chain_class.loc[(after, m)] - chain_class.loc[(before, m)]
        step_rows.append({"stage": after, "kind": KIND[after], "model": m,
                          "removed": len(stages.get(before, [])) - len(stages.get(after, [])),
                          "change_mean": change.mean(), "change_std": change.std(),
                          "folds_higher": int((change > 0).sum()),
                          "within_fold_std": abs(change.mean()) < chain_table.loc[before, f"{m} f1_std"],
                          "classes_better": int((class_change > 0.01).sum()),
                          "classes_worse": int((class_change < -0.01).sum()),
                          "ari_higher_at_k": int((cluster_wide[after] > cluster_wide[before]).sum())})
chain_steps = pd.DataFrame(step_rows)
# The selected subset against the raw features, fold by fold: new-recording macro F1 and CPU time
chain_new = chain.pivot_table(index="fold", columns=["model", "stage"], values="new_recording_f1")
chain_time = chain.pivot_table(index="fold", columns=["model", "stage"], values="seconds")
new_gain = pd.DataFrame({m: chain_new[(m, COPIES)] - chain_new[(m, stage_names[0])] for m in MODELS})
# Folds in which the selected subset is faster than the raw features and than the first stage without timing
faster_than = {(m, s): int((chain_time[(m, COPIES)] < chain_time[(m, s)]).sum()) for m in MODELS for s in stage_names[:2]}
chain_steps.loc[chain_steps["stage"] == COPIES, "removed"] = np.nan  # differs by fold (about 10)
leak_steps = chain_steps[chain_steps["kind"] == "leakage"]
capture_part = leak_steps.groupby("model")["change_mean"].sum()  # macro F1 removed at the leakage stages
other_part = chain_steps[chain_steps["kind"] != "leakage"].groupby("model")["change_mean"].sum()
timing_loss = chain_class.loc[(stage_names[0], "Random Forest")] - chain_class.loc[(stage_names[1], "Random Forest")]
first_clean = stage_names[1]  # the first stage without the capture timing
# Adjusted Rand index refitted on each training fold: mean over k, one value per fold and stage
fold_ari = pd.DataFrame(chain_fold_clusters).groupby(["stage", "fold"])["ari"].mean().unstack("fold").loc[stage_names]
fold_ari_gain = fold_ari.loc[COPIES] - fold_ari.loc[stage_names[0]]  # selected subset minus raw features, per fold
fold_ari_steps = {after: int((fold_ari.loc[after] > fold_ari.loc[before]).sum())  # folds where a stage raises the ARI
                  for before, after in zip(stage_names[:-1], stage_names[1:])}

print("======== Before ========")
print("The selection replayed from the raw features, one removal at a time: "
      + " -> ".join(f"{name} ({len(stages.get(name, selected))})" for name in stage_names))
print(f"Check: the exported set rebuilt from raw values differs from the exported values by at most {rebuilt_gap:.4f} "
      "(standardised units), so the replay reproduces the cleaning")

print("\n======== After ========")
print("1. Table: per stage, the kind of removal, the features, macro F1 of both models (mean and spread over the folds), "
      "macro F1 with the capture features shuffled (new_f1), CPU seconds per fold, the K-Means agreement with the "
      "classes on all training rows (ari, purity), and the cross-recording macro F1 (cross_f1, below)")
display(chain_table.round(4))
print("2. Table: each removal against the stage before it, on the same validation rows (paired; within_fold_std = "
      "the change is smaller than one fold spread)")
display(chain_steps.round(4))
print("3. Summed over the leakage stages, macro F1 changes by " + ", ".join(f"{m} {capture_part[m]:+.4f}" for m in MODELS)
      + "; over the other stages by " + ", ".join(f"{m} {other_part[m]:+.4f}" for m in MODELS))
print("   Random Forest classes that lose most F1 when the timing feature goes: "
      + ", ".join(f"{c} {v:.3f}" for c, v in timing_loss.nlargest(3).items()))
print("4. Capture dependence (the capture features shuffled together in each validation fold, so they say nothing "
      "about the attack): selected subset minus raw features "
      + "; ".join(f"{m} {new_gain[m].mean():+.4f} (higher in {(new_gain[m] > 0).sum()} of {FA_FOLDS} folds)" for m in MODELS))
print("5. CPU time: " + "; ".join(
    f"{m} raw {chain_table.loc[stage_names[0], f'{m} seconds']:.2f}, {first_clean.split(' (')[0].lower()} "
    f"{chain_table.loc[first_clean, f'{m} seconds']:.2f}, selected {chain_table.loc[COPIES, f'{m} seconds']:.2f} s per fold "
    f"(faster than raw in {faster_than[(m, stage_names[0])]} and than {first_clean.split(' (')[0].lower()} in "
    f"{faster_than[(m, stage_names[1])]} of {FA_FOLDS} folds)" for m in MODELS))
print("   CPU time of the selected subset, change against the raw features and against the first stage without timing: "
      + "; ".join(f"{m} {chain_table.loc[COPIES, f'{m} seconds'] / chain_table.loc[stage_names[0], f'{m} seconds'] - 1:+.1%} and "
                  f"{chain_table.loc[COPIES, f'{m} seconds'] / chain_table.loc[first_clean, f'{m} seconds'] - 1:+.1%}" for m in MODELS))
print(f"6. K-Means refitted on each training fold ({FOLD_STARTS} starts), adjusted Rand index (mean over k) per stage: "
      + "; ".join(f"{name.split(' (')[0]} {fold_ari.loc[name].mean():.3f} +/- {fold_ari.loc[name].std():.3f}" for name in stage_names))
print(f"   Selected subset minus raw features, per fold: {', '.join(f'{g:+.3f}' for g in fold_ari_gain)} | "
      f"(mean {fold_ari_gain.mean():+.3f} +/- {fold_ari_gain.std():.3f}) | "
      "folds where each removal raises it: " + ", ".join(f"{name.split(' (')[0]} {n}" for name, n in fold_ari_steps.items()))
print(f"7. Table: cross-recording test, macro F1 over the {len(shift_classes)} classes recorded in both small batches, "
      "trained with one recording and tested on the other (columns: model, recording held out)")
display(cross_table.loc[stage_names].round(3))
print("8. Check of step viii, the exported features without flow_duration on another recording: "
      + "; ".join(f"{m} {cross_table.loc[NO_FLOW, m].mean():.3f} (with it {cross_table.loc[stage_names[3], m].mean():.3f})"
                  for m in MODELS))
exported_cross = cross_table.loc[stage_names[3]]  # the exported features, per model and recording
print(f"9. Control (is the test true by construction?): change in cross-recording macro F1 against the exported features. "
      f"Removing the kept features that shift most between the recordings, unevenly ({', '.join(SHIFTING)}): "
      + ", ".join(f"{m} {(cross_table.loc[NO_SHIFTING, m] - exported_cross[m]).mean():+.3f}" for m in MODELS)
      + " | adding back one stand-in: "
      + "; ".join(f"{f} " + ", ".join(f"{m} {(cross_table.loc[f'Exported plus {f}', m] - exported_cross[m]).mean():+.3f}"
                                      for m in MODELS) for f in STAND_INS))

# 10. What each phase contributes: raw features -> exported features (cleaning) -> selected subset (Phase 2)
cleaned_stage = stage_names[3]  # the exported features, the end of the cleaning phase
phase_rows, phase_better = {}, {}
for m in MODELS:
    copies_step = chain_steps[(chain_steps["stage"] == COPIES) & (chain_steps["model"] == m)].iloc[0]
    phase_rows[f"{m} macro F1"] = chain_table[f"{m} f1_mean"]
    phase_better[f"{m} macro F1"] = f"{copies_step.folds_higher} of {FA_FOLDS} folds"
    phase_rows[f"{m} macro F1, another recording"] = chain_table[f"{m} cross_f1"]
    phase_better[f"{m} macro F1, another recording"] = (
        f"{int((cross_table.loc[COPIES, m] > cross_table.loc[cleaned_stage, m]).sum())} of 2 recordings")
    phase_rows[f"{m} CPU seconds per fold"] = chain_table[f"{m} seconds"]
    phase_better[f"{m} CPU seconds per fold"] = (
        f"{int((chain_time[(m, COPIES)] < chain_time[(m, cleaned_stage)]).sum())} of {FA_FOLDS} folds faster")
phase_rows["K-Means ARI, all rows"] = chain_table["ari"]
phase_better["K-Means ARI, all rows"] = f"{copies_step.ari_higher_at_k} of {len(K_RANGE)} k"
phase_rows["K-Means ARI, refitted per fold"] = fold_ari.mean(axis=1)
phase_better["K-Means ARI, refitted per fold"] = f"{fold_ari_steps[COPIES]} of {FA_FOLDS} folds"
phase_table = pd.DataFrame(phase_rows).T[[stage_names[0], cleaned_stage, COPIES]]
phase_table.columns = ["raw features", "after cleaning", "after Phase 2"]
phase_table["cleaning change"] = phase_table["after cleaning"] - phase_table["raw features"]
phase_table["Phase 2 change"] = phase_table["after Phase 2"] - phase_table["after cleaning"]
phase_table["Phase 2 better in"] = pd.Series(phase_better)
print("10. Table: what each phase contributes (raw features -> exported features after cleaning -> selected subset after "
      "Phase 2); the last column counts where the Phase 2 step is better (CPU: lower is better)")
display(phase_table.round(4))

# 11. Split-half test: the stand-in rules of step x.2 rerun with half of the classes recorded twice, and the result
#     scored on the other half, which the rules never saw (the cross-recording test above reuses the rows that found them)
screened = stages[stage_names[2]]  # the features the stand-in rules screened
filled = raw_values[screened].fillna(raw_values[screened].median())  # median-filled, as in step ix
labels_raw = pd.Series(y_fa.to_numpy(), index=raw_values.index)  # the same rows in the same order (step 3 check)
halves = [sorted(shift_classes)[0::2], sorted(shift_classes)[1::2]]  # every other class
half_rows, half_found = [], []
for half_no, (found_on, scored_on) in enumerate([(halves[0], halves[1]), (halves[1], halves[0])]):
    seen = ~labels_raw.isin(scored_on).to_numpy()  # every row except the scored classes
    rows, labels = filled[seen], labels_raw[seen]
    stand = batch_stand_ins(rows, raw_values["iat"][seen])[0]
    rest = [c for c in screened if c not in stand]
    # The recordings come from all rows: with half of the classes left out, a small recording can fall below 1%
    shift_here, _ = recording_shift(rows[rest], pd.Series(recording, index=raw_values.index)[seen], labels)
    shifted_here = shift_here.index[shift_here["follows recording"]].to_list()
    hidden_here = []
    if stand or shifted_here:  # a hidden stand-in carries any stand-in found so far, by either rule
        products_here = hidden_stand_ins(rows[[c for c in rest if c not in shifted_here]], rows[stand + shifted_here])
        hidden_here = products_here.loc[products_here["carries stand-in"], "feature"].unique().tolist()
    found = stand + shifted_here + hidden_here
    half_found.append(found)
    scored = labels_raw.isin(scored_on).to_numpy()
    for held in sorted(pd.Series(recording).value_counts().index[1:3]):  # the two small recordings
        test_pos = np.where(scored & (recording == held))[0]
        train_pos = np.where(~(scored & (recording == held)))[0]
        for set_name, cols in (("with the stand-ins", screened), ("without them", [c for c in screened if c not in found])):
            X_tr, X_te = stage_fold(cols, train_pos, test_pos)
            for m, make in MODELS.items():
                predicted = fit_model(make, X_tr, y_fa.iloc[train_pos]).predict(X_te)
                half_rows.append({"half": half_no + 1, "held_out": held, "features": set_name, "model": m,
                                  "macro_f1": f1_score(y_fa.iloc[test_pos], predicted, average="macro",
                                                       labels=sorted(scored_on))})
half_table = pd.DataFrame(half_rows).pivot_table(index=["half", "features"], columns="model", values="macro_f1")
print(f"11. Split-half test: the stand-in rules rerun on half of the {len(shift_classes)} classes recorded twice, "
      "then scored on the other half")
for half_no, found in enumerate(half_found, start=1):
    print(f"   Half {half_no}: rules found {', '.join(found)} (the same as with all classes: {set(found) == set(STAND_INS)})")
print("   Table: macro F1 on another recording for the classes the rules never saw (mean of the two recordings)")
display(half_table.round(3))

# ======== Conclusion ========
print("\n======== Conclusion ========")
quiet = bool(chain_steps.loc[chain_steps["kind"] != "leakage", "within_fold_std"].all())  # other stages within noise
print(f"Most of the score the selection removes comes from the leakage stages, the other stages "
      f"{'cost no measurable' if quiet else 'change the'} macro F1, and with the capture features carrying no information "
      f"the selected subset scores "
      f"{'higher' if (new_gain > 0).all().all() else 'not always higher'} than the raw features for both models. "
      f"On another recording the raw features score {chain_table.loc[stage_names[0], 'Random Forest cross_f1']:.3f} "
      f"(Random Forest) against {chain_table.loc[COPIES, 'Random Forest cross_f1']:.3f} for the selected subset.")

# ======== Figures ========
x = np.arange(len(stage_names))
ticks = [f"{name.split(' (')[0]}\n{chain_table.loc[name, 'features']:.0f} features" for name in stage_names]
with plt.style.context(STYLE):
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(10, 3.4))
    for m in MODELS:  # one line per model, error bars = fold standard deviation
        ax1.errorbar(x, chain_table[f"{m} f1_mean"], yerr=chain_table[f"{m} f1_std"], marker="o", ms=3, capsize=2, label=m)
        ax2.plot(x, chain_table[f"{m} seconds"], marker="o", ms=3, label=m)
    ax3.plot(x, chain_table["ari"], marker="o", ms=3, label="Adjusted Rand index")
    ax3.plot(x, chain_table["purity"], marker="o", ms=3, label="Purity")
    ax1.set(title="Cross-validated macro F1", ylabel="Macro F1")
    ax2.set(title="Fit and predict CPU time", ylabel="CPU seconds per fold")
    ax3.set(title="K-Means agreement with the classes", ylabel="Mean over k")
    for ax in (ax1, ax2, ax3):
        ax.set_xticks(x, ticks, rotation=45, ha="right", fontsize=6)
        ax.legend(fontsize=7)
        ax.grid(linestyle="--", alpha=0.3)
    fig.tight_layout()
display(finish_fig(fig, "fig_fa_chain"))

### 7. Final Selection and Export

- **Problem:** the next phases need the selected subset and the reason for every removal.
- **Decision:** apply the rule to all training rows, print every left-out feature's reason and the `SELECTED` list; leave the cleaned files unchanged.
- **Alternatives:** overwriting the cleaned files would lose the comparison with all features.

In [ ]:
# ======== Cleaning: a reason for every feature left out ========
corr_all = X_fa.corr(method="spearman").abs()
chosen_limit = float(chosen.split()[-1]) if "Corr" in chosen else None  # e.g. "Relevant + Corr 0.90" -> 0.90
reason, partner_of, redundant = {}, {}, set()
for f in X_fa.columns.difference(selected):
    partner = corr_all.loc[f, selected].idxmax()  # the most similar selected feature
    partner_of[f] = partner
    # Under a relevance rule a left-out feature may add nothing over noise; under a correlation rule it may be redundant
    if chosen.startswith("Relevant") and f not in final_relevant:
        reason[f] = f"not relevant (permutation drop {final_drops[f]:.4f}, noise band up to {probe_band.max():.4f})"
    elif chosen_limit is not None and corr_all.loc[f, partner] >= chosen_limit:
        reason[f] = f"redundant with {partner} (|rho| {corr_all.loc[f, partner]:.2f})"
        redundant.add(f)
    else:
        reason[f] = "outside the most important features"
# Insight: with the copies gone, does the importance of the kept partner rise? Same split and forest as step 3
rf_selected = forest().fit(X_fit[selected], y_fit)
perm_sel = permutation_importance(rf_selected, X_val[selected], y_val, scoring="f1_macro", n_repeats=PERM_REPEATS,
                                  random_state=SEED, n_jobs=1)  # the forest itself already uses CORES cores
partners = sorted({partner_of[f] for f in redundant})  # selected features that a removed feature copies
# Permutation drop of each partner: among all features against the selected subset
importance_shift = pd.DataFrame({"all_features": perm_drop[partners],
                                 "selected_subset": pd.Series(perm_sel.importances_mean, index=selected)[partners]})

# Stability: in how many training folds the chosen rule kept each feature, and the overlap with the exported list
fold_kept = [f for lists in fold_lists for f in lists[chosen]]
kept_in_folds = pd.Series(fold_kept).value_counts().reindex(X_fa.columns, fill_value=0)
jaccard = []
for lists in fold_lists:  # shared features divided by all features of the two lists
    fold_set, final_set = set(lists[chosen]), set(selected)
    jaccard.append(len(fold_set & final_set) / len(fold_set | final_set))
jaccard = pd.Series(jaccard)
selection = importance.assign(kept_in_folds=kept_in_folds, selected=importance.index.isin(selected),
                              reason=pd.Series(reason))  # one row per feature
selection.index.name = "feature"

# ======== Conclusion ========
swapped = selection.index[~selection["selected"] & (selection["kept_in_folds"] > 0)]  # kept in a fold, not exported
print("======== Before ========")
print(f"Chosen rule: {chosen}, applied once to all training rows (step 5)")

print("\n======== After ========")
print(f"Selected: {len(selected)} of {X_fa.shape[1]} features | per family: "
      + ", ".join(f"{fam} {n}" for fam, n in selection.loc[selection["selected"], "family"].value_counts().items()))
print("Left out, and why:")
for f, why in reason.items():
    print(f"  {f}: {why}")
print(f"Stability: overlap of each fold's list with the exported list (Jaccard) {', '.join(f'{j:.3f}' for j in jaccard)} "
      f"(mean {jaccard.mean():.3f}); "
      f"{(kept_in_folds[selected] == FA_FOLDS).sum()} of {len(selected)} selected features kept in every fold")
print("Swaps between folds (a near-copy kept instead): "
      + "; ".join(f"{f} in {kept_in_folds[f]} folds ({reason[f]})" for f in swapped))
print(f"Relevance on all training rows: random columns drop the macro F1 by {probe_band.min():.4f} to {probe_band.max():.4f} "
      f"(the noise band); not above it: {', '.join(f for f in X_fa.columns if f not in final_relevant)}")
print("Table: permutation importance of the features that had near-copies, among all features and in the selected subset")
display(importance_shift.round(4))

print("\n======== Conclusion ========")
print(f"The selected subset ({len(selected)} features), ready to copy into the modelling phases (the cleaned files "
      f"keep all {X_fa.shape[1]} features, so both can be compared):")
print(f"SELECTED = {sorted(selected)}")

### Findings of Phase 2

Every number is printed above.

- **The whole selection (step 6).** Most of the score the selection removes is removed at the leakage stages; the constant features and the redundant copies go without a measurable loss. With the capture features shuffled, the raw features score well below the selected subset, which uses none of them; trained with one recording and tested on another, the raw features score almost nothing while the selected subset keeps part of its score (the same two recordings defined the stand-ins, but a control shows the gain is not true by construction: removing the kept features that shift most between the recordings changes little, while adding back any stand-in costs much more, step 6 item 9); and refitted on each fold the cluster agreement of K-Means is higher for the selected subset than for the raw features in every fold.
- **Leakage.** No kept feature or pair of strong features reaches the limit; `iat` does on its own, which supports its removal in step viii.
- **Importance.** The TCP-flag and traffic-volume families matter most; inside the attack groups, floods and scans rest on the flags and the other attacks on packet size (the groups with few rows are weaker evidence).
- **Subset.** The chosen correlation filter keeps every family, keeps the macro F1 of all features within the fold noise and needs less CPU time; cutting by Gini importance alone loses macro F1.
- **Passed on from cleaning.** Removing `duration` or `protocol_type` changes the macro F1 by less than one fold spread, so both stay; removing `flow_duration` (kept in step viii) does not raise the score on another recording (step 6, item 8), so it stays too.

### Handoff to the Next Phases

Phases 1 and 2 end here. Checklist for the supervised and unsupervised phases:

- **Files.** `cleaned_train.csv` (real training rows only) and `cleaned_test.csv` (new inputs only) hold the kept features, already log-transformed and scaled, plus `label` as text. Every value was fitted on the training part only, so use the test file once, for the final evaluation.
- **Selected features.** Copy the `SELECTED` list printed at the end of Phase 2 (step 7, which also gives every left-out feature its reason). Train on the selected subset and on all features and report both.
- **Class imbalance.** Use `class_weight="balanced"` for the tree models (scikit-learn computes the weights shown in step xiv from the training labels); for KNN, apply SMOTE inside each training fold with `prepare_fold(train_idx, val_idx)` (step xv), never before the split or on the test set.
- **Cross-validation.** `prepare_fold()` rebuilds a fold from raw values and fits every value on the training fold only. Flood flows repeat across folds (step xv), so flood scores stay somewhat optimistic.
- **Expect confusion** between DoS and DDoS floods of the same kind: they differ mainly in the number of sending machines, which one flow record does not hold, and some of their flows have identical inputs (step x.3). Classes with few rows score lowest.
- **Evaluation.** Report accuracy with macro precision, recall, F1 and a confusion matrix, and compare training with validation scores to spot over- and underfitting.
- **Limitations.** No per-session identifier exists, and the data holds no benign traffic, so the models label attack types rather than detect attacks.
- **Numbers.** Every number the report quotes comes from this notebook's run: printed above, or computed from printed values by `scripts/report_tables.py`.